# Ara25 — mT5

Public research-preview notebook. Outputs and volatile metadata were removed. Set local dataset/output paths in the configuration cells before running.


In [ ]:
!pip install -U transformers datasets rouge-score nltk

In [ ]:
!pip install -q transformers datasets evaluate rouge-score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q evaluate

In [ ]:
!pip install -q rouge-score

In [ ]:
!pip install -q sacrebleu

In [ ]:
# -*- coding: utf-8 -*-
# Patch-FT (8 epochs + EarlyStopping on ROUGE-Lsum + tuned LR/MAX_IN/MAX_OUT/Batch + cosine)
import os, json, gc, re, random, pandas as pd, torch, numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback, GenerationConfig
)
from inspect import signature
import evaluate

# ---------------- Paths & Config ----------------
JSONL = "/content/drive/MyDrive/PACKED2x2_6144/train_ready_6144.jsonl"
OUT   = "/content/drive/MyDrive/mt5_xlsum444_patch_6144"
BASE  = "csebuetnlp/mT5_multilingual_XLSum"
AR_PREFIX = "لخص بالعربية الفصحى بدقة واختصار ووضوح:"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
device = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ---------------- Load data ----------------
recs = [json.loads(l) for l in open(JSONL, "r", encoding="utf-8") if l.strip()]
df   = pd.DataFrame(recs)
ds   = Dataset.from_pandas(df)
spl  = ds.train_test_split(test_size=0.10, seed=SEED)
tr, va = spl["train"], spl["test"]

# ---------------- Cleaning helpers ----------------
_dup_open_re = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+')
def clean_merged(s: str) -> str:
    return _dup_open_re.sub(r'\1', s or "")
def ensure_prefix(s: str) -> str:
    s = s or ""
    t = s.lstrip()
    return s if t.startswith(AR_PREFIX) else (AR_PREFIX + " " + s)

# ---------------- Tokenizer / Model ----------------
tok = AutoTokenizer.from_pretrained(BASE, use_fast=False)
mdl = AutoModelForSeq2SeqLM.from_pretrained(BASE)
mdl.config.use_cache = False  # لازم للتدرّج + checkpointing

# ---------------- Tokenization ----------------
MAX_IN, MAX_OUT = 6144, 512
def tok_batch(batch):
    ins_texts = [ensure_prefix(clean_merged(x)) for x in batch["Merged_Articles"]]
    model_inputs = tok(ins_texts, truncation=True, max_length=MAX_IN)
    labels = tok(text_target=batch["target_summary"], truncation=True, max_length=MAX_OUT)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tr = tr.map(tok_batch, batched=True, remove_columns=tr.column_names)
va = va.map(tok_batch, batched=True, remove_columns=va.column_names)

coll = DataCollatorForSeq2Seq(tokenizer=tok, model=mdl, pad_to_multiple_of=8)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

# ---------------- Metrics (ROUGE) ----------------
rouge = evaluate.load("rouge")

UNK_ID = tok.unk_token_id if tok.unk_token_id is not None else (tok.pad_token_id if tok.pad_token_id is not None else 0)
VOCAB_SIZE = tok.vocab_size

def _clip_ids(arr, replace_id=UNK_ID):
    import numpy as _np
    arr = _np.asarray(arr)
    mask = (arr < 0) | (arr >= VOCAB_SIZE)
    if _np.any(mask):
        arr = arr.copy()
        arr[mask] = replace_id
    return arr

# تنعيم بسيط للنص قبل الاحتساب (لا يغيّر التدريب)
import re as _re
def _tidy(s: str) -> str:
    if not s: return s
    s = _re.sub(r"\s+", " ", s)
    s = s.replace(" ،","،").replace(" .",".")
    return s.strip()

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    preds  = _clip_ids(preds, UNK_ID)
    labels = np.where(labels == -100, tok.pad_token_id, labels)
    labels = _clip_ids(labels, tok.pad_token_id)

    decoded_preds  = [_tidy(p) for p in tok.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [_tidy(l) for l in tok.batch_decode(labels, skip_special_tokens=True)]

    res = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    # نعيد مفاتيح ROUGE مباشرة (الـ Trainer سيضيف eval_ تلقائياً)
    return {k: float(res[k]) for k in res.keys()}

# ---------------- Training Args ----------------
sig_args = signature(Seq2SeqTrainingArguments.__init__).parameters
eval_kw = {}
if "eval_strategy" in sig_args:
    eval_kw["eval_strategy"] = "epoch"
elif "evaluation_strategy" in sig_args:
    eval_kw["evaluation_strategy"] = "epoch"

# نحاول دعم processing_class إن كان متاحاً (لإزالة التحذير)
accepts_processing = "processing_class" in signature(Seq2SeqTrainer.__init__).parameters

args = Seq2SeqTrainingArguments(
    output_dir=OUT,
    num_train_epochs=8,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.2,

    optim="adafactor",
    weight_decay=0.01,
    max_grad_norm=1.0,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    eval_accumulation_steps=1,

    predict_with_generate=True,
    generation_max_length=MAX_OUT,
    generation_num_beams=4,

    save_strategy="epoch",
    save_total_limit=3,               # إبقاء آخر 3 فقط
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,

    logging_steps=20,
    logging_first_step=True,
    report_to=[],

    fp16=(not use_bf16),
    bf16=use_bf16,
    group_by_length=True,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    save_safetensors=True,
    **eval_kw,
)

# ---------------- Trainer ----------------
trainer_kwargs = dict(
    model=mdl,
    args=args,
    train_dataset=tr,
    eval_dataset=va,
    data_collator=coll,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=1e-4)],
)
if accepts_processing:
    trainer_kwargs["processing_class"] = tok  # يزيل تحذير deprecation
else:
    trainer_kwargs["tokenizer"] = tok

trainer = Seq2SeqTrainer(**trainer_kwargs)

# ---------------- Train ----------------
trainer.train()

# ---------------- Save best model + generation config ----------------
best_dir = f"{OUT}/final_bestx2"
trainer.save_model(best_dir)
# احفظ إعدادات التوليد المطابقة للتدريب للاستخدام لاحقًا
GenerationConfig(
    num_beams=4, do_sample=False, max_new_tokens=MAX_OUT, early_stopping=True
).save_pretrained(best_dir)

# حفظ التوكنيزر/المعالج
if accepts_processing:
    tok.save_pretrained(best_dir)
else:
    tok.save_pretrained(best_dir)

print("✅ Saved best model to:", best_dir)

# تنظيف GPU
del tr, va, ds, df, recs; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# =================== One-Cell: Generate + Evaluate (multi-metric, JSONL with Ref1/Ref2) ===================
!pip -q install "transformers==4.44.2" "evaluate==0.4.2" "rouge_score==0.1.2" "sacrebleu==2.4.0" "bert-score==0.3.13" "moverscore==1.0.3" tqdm

import os, json, re, random, gc, math, sys
from datetime import datetime
from typing import List, Dict, Any
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
import evaluate

# --------- Paths ---------
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"  # JSONL الجديد
BEST_DIR  = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/final_bestx2"
OUT_DIR   = f"/content/drive/MyDrive/mt5_xlsum444_patch_6144/test_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

# --------- Env ---------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# =================== Robust Loader (JSON/JSONL) ===================
def load_any_json_or_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s: continue
            if s.startswith("```"): continue
            if s.startswith("`"): s = s[1:]
            s = s.rstrip("`,")
            if "{" in s or "[" in s:
                cut_positions = [i for i in (s.find("{"), s.find("[")) if i != -1]
                if cut_positions:
                    i0 = min(cut_positions)
                    if i0 > 0: s = s[i0:]
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj); continue
                elif isinstance(obj, list): recs.extend([x for x in obj if isinstance(x, dict)]); continue
            except json.JSONDecodeError:
                pass
    if recs: return recs
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        text = f.read().strip()
    if text.startswith("```"):
        parts = text.split("```")
        for blk in parts:
            if "{" in blk or "[" in blk:
                text = blk.strip(); break
    text = text.lstrip("`").rstrip("`")
    data = json.loads(text)
    if isinstance(data, dict): return [data]
    if isinstance(data, list): return [x for x in data if isinstance(x, dict)]
    raise ValueError("Unsupported JSON structure in test file.")

# =================== Preprocessing (match training) ===================
AR_PREFIX = "لخص بالعربية الفصحى بدقة واختصار ووضوح:"
_dup_open_re = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+')

def clean_merged(s: str) -> str:
    return _dup_open_re.sub(r'\1', s or "")

def ensure_prefix(s: str) -> str:
    s = s or ""
    return s if s.lstrip().startswith(AR_PREFIX) else (AR_PREFIX + " " + s)

def tidy(s: str) -> str:
    if not s: return s
    s = re.sub(r"\s+", " ", s); s = s.replace(" ،","،").replace(" .",".")
    return s.strip()

def extract_input_and_refs(rec: Dict[str, Any]):
    # الحقول الجديدة: Merged_Articles + target_summary (Ref1) + target_summary_2 (Ref2)
    inp = rec.get("Merged_Articles", None)
    inp = ensure_prefix(clean_merged(inp)) if isinstance(inp, str) else None
    refs = []
    # الترتيب مهم: Ref1 ثم Ref2
    for k in ["target_summary", "target_summary_2"]:
        v = rec.get(k, None)
        if isinstance(v, str) and v.strip():
            refs.append(tidy(v))
    return inp, refs

# =================== Load model/tokenizer ===================
tok = AutoTokenizer.from_pretrained(BEST_DIR, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR).to(device).eval()
try:
    gen_cfg = GenerationConfig.from_pretrained(BEST_DIR)
except Exception:
    gen_cfg = GenerationConfig(num_beams=4, do_sample=False, max_new_tokens=512, early_stopping=True)

# ==== (FIX) Ensure decoder_start_token_id for T5/mT5 ====
pad_id = tok.pad_token_id
if pad_id is None:
    tok.add_special_tokens({'pad_token': '<pad>'})
    model.resize_token_embeddings(len(tok))
    pad_id = tok.pad_token_id
if getattr(model.config, "decoder_start_token_id", None) is None:
    model.config.decoder_start_token_id = pad_id
if getattr(gen_cfg, "decoder_start_token_id", None) is None:
    gen_cfg.decoder_start_token_id = pad_id
if getattr(model.config, "eos_token_id", None) is None and tok.eos_token_id is not None:
    model.config.eos_token_id = tok.eos_token_id

# =================== Read test, generate ===================
raw = load_any_json_or_jsonl(TEST_PATH)
print(f"Loaded test items: {len(raw)}")
MAX_INPUT = 6144
BATCH_SIZE = 1

records = []
for i, rec in enumerate(raw):
    inp, refs = extract_input_and_refs(rec)
    records.append({
        "id": rec.get("id", i),
        "title": rec.get("title", ""),
        "input": inp,
        "references": refs,
        "prediction": None
    })

todo = [i for i,r in enumerate(records) if r["input"]]
def batched(idx_list, n):
    for i in range(0, len(idx_list), n):
        yield idx_list[i:i+n]

for chunk in tqdm(list(batched(todo, BATCH_SIZE)), desc="Generating"):
    texts = [records[i]["input"] for i in chunk]
    enc = tok(texts, return_tensors="pt", truncation=True, max_length=MAX_INPUT, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**enc, generation_config=gen_cfg)
    dec = tok.batch_decode(out, skip_special_tokens=True)
    for j, idx in enumerate(chunk):
        records[idx]["prediction"] = tidy(dec[j])

preds = [r["prediction"] or "" for r in records]

# =================== Prepare references (multi / per-ref) ===================
refs_per_ex = []
ref1_only = []
ref2_only = []
has_ref2_mask = []
for r in records:
    refs = r["references"][:] if r["references"] else [""]
    refs_per_ex.append(refs)
    # Ref1 always first if exists
    ref1_only.append([refs[0] if len(refs) >= 1 else ""])
    # Ref2 if exists else empty
    if len(refs) >= 2 and str(refs[1]).strip():
        ref2_only.append([refs[1]])
        has_ref2_mask.append(True)
    else:
        ref2_only.append([""])
        has_ref2_mask.append(False)

num_has_r2 = sum(has_ref2_mask)
print(f"Ref1 count: {len(ref1_only)} | Ref2 non-empty: {num_has_r2}/{len(records)}")

# =================== ROUGE ===================
rouge_metric = evaluate.load("rouge")

def rouge_pair(pred, ref):
    sc = rouge_metric.compute(predictions=[pred], references=[ref], use_stemmer=False)
    return {k: float(sc[k]) for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

# Per-ref
r1_rouge = []; r2_rouge = []; best_ref_choice = []
for i, r in tqdm(list(enumerate(records)), total=len(records), desc="Scoring (ROUGE)"):
    pred = preds[i]
    refs = r["references"] if r["references"] else [""]
    # Ref1
    sc1 = rouge_pair(pred, refs[0])
    r1_rouge.append(sc1)
    # Ref2 if any
    if len(refs) >= 2 and refs[1].strip():
        sc2 = rouge_pair(pred, refs[1]); r2_rouge.append(sc2)
    else:
        r2_rouge.append({"rouge1": np.nan, "rouge2": np.nan, "rougeL": np.nan, "rougeLsum": np.nan})
    # Best-over-refs by RougeLsum
    cand = [(sc1["rougeLsum"], refs[0])]
    if len(refs) >= 2 and refs[1].strip():
        cand.append((r2_rouge[-1]["rougeLsum"], refs[1]))
    best_ref_choice.append(max(cand, key=lambda x: x[0])[1])

def mean_rouge(rows, key):
    arr = [x[key] for x in rows]
    return float(np.nanmean(arr)) if np.any(~np.isnan(arr)) else None

# =================== BLEU / chrF ===================
sacrebleu_metric = evaluate.load("sacrebleu")
# multi-ref
sacrebleu = sacrebleu_metric.compute(predictions=preds, references=refs_per_ex) if refs_per_ex else {"score": None}
# single-ref (Ref1) on full set
bleu_ref1 = sacrebleu_metric.compute(predictions=preds, references=ref1_only)["score"] if records else None
# single-ref (Ref2) on subset with real Ref2
if num_has_r2 > 0:
    idx2 = [i for i, ok in enumerate(has_ref2_mask) if ok]
    preds2 = [preds[i] for i in idx2]
    refs2  = [[records[i]["references"][1]] for i in idx2]
    bleu_ref2 = sacrebleu_metric.compute(predictions=preds2, references=refs2)["score"]
else:
    bleu_ref2 = None

# chrF
try:
    chrf_metric = evaluate.load("chrf")
    chrf = chrf_metric.compute(predictions=preds, references=refs_per_ex) if refs_per_ex else {"score": None}
    chrf_ref1 = chrf_metric.compute(predictions=preds, references=ref1_only)["score"] if records else None
    if num_has_r2 > 0:
        chrf_ref2 = chrf_metric.compute(predictions=preds2, references=refs2)["score"]
    else:
        chrf_ref2 = None
except Exception:
    chrf = {"score": None}; chrf_ref1 = None; chrf_ref2 = None

# =================== METEOR (per-ref + best) ===================
try:
    meteor_metric = evaluate.load("meteor")
    meteor_ref1 = float(meteor_metric.compute(predictions=preds, references=[x[0] for x in ref1_only]).get("meteor", np.nan))
    if num_has_r2 > 0:
        meteor_ref2 = float(meteor_metric.compute(predictions=preds2, references=[x[0] for x in refs2]).get("meteor", np.nan))
    else:
        meteor_ref2 = None
    meteor_best = float(meteor_metric.compute(predictions=preds, references=best_ref_choice).get("meteor", np.nan))
except Exception as e:
    print("METEOR skipped:", e)
    meteor_ref1 = meteor_ref2 = meteor_best = None

# =================== BERTScore (Arabic) — per-ref + best-over-refs ===================
bertscore = {"ref1": None, "ref2": None, "best": None}
try:
    bertscore_metric = evaluate.load("bertscore")
    bs_batch = 8

    # Ref1
    res1 = bertscore_metric.compute(
        predictions=preds, references=[x[0] for x in ref1_only],
        lang="ar", model_type="xlm-roberta-large",
        rescale_with_baseline=True, batch_size=bs_batch, device=device
    )
    bertscore["ref1"] = {
        "P": float(np.mean(res1["precision"])),
        "R": float(np.mean(res1["recall"])),
        "F1": float(np.mean(res1["f1"])),
    }

    # Ref2 (subset)
    if num_has_r2 > 0:
        res2 = bertscore_metric.compute(
            predictions=preds2, references=[x[0] for x in refs2],
            lang="ar", model_type="xlm-roberta-large",
            rescale_with_baseline=True, batch_size=bs_batch, device=device
        )
        bertscore["ref2"] = {
            "P": float(np.mean(res2["precision"])),
            "R": float(np.mean(res2["recall"])),
            "F1": float(np.mean(res2["f1"])),
        }
    else:
        bertscore["ref2"] = None

    # best-over-refs
    # ابني أعمدة مرجعية لكل i_ref ثم اختر الأفضل لكل عنصر
    def ref_column(i_ref: int):
        col = []
        for refs in refs_per_ex:
            col.append(refs[i_ref] if i_ref < len(refs) else refs[0])
        return col
    ncols = max((len(r) for r in refs_per_ex), default=1)
    per_ref_scores = []
    for i_ref in range(ncols):
        refs_i = ref_column(i_ref)
        res = bertscore_metric.compute(
            predictions=preds, references=refs_i,
            lang="ar", model_type="xlm-roberta-large",
            rescale_with_baseline=True, batch_size=bs_batch, device=device
        )
        per_ref_scores.append(res)
    P_list, R_list, F_list = [], [], []
    for j in range(len(records)):
        best_idx = 0; best_f = -1.0
        for i_ref in range(len(per_ref_scores)):
            f = per_ref_scores[i_ref]["f1"][j]
            if f > best_f: best_f, best_idx = f, i_ref
        P_list.append(per_ref_scores[best_idx]["precision"][j])
        R_list.append(per_ref_scores[best_idx]["recall"][j])
        F_list.append(per_ref_scores[best_idx]["f1"][j])
    bertscore["best"] = {"P": float(np.mean(P_list)), "R": float(np.mean(R_list)), "F1": float(np.mean(F_list))}
except Exception as e:
    print("BERTScore skipped:", e)

# =================== MoverScore (per-ref + best) ===================
ms_results = {"ref1": None, "ref2": None, "best": None}
try:
    ms_metric = evaluate.load("moverscore")
    # Ref1 (full)
    ms_ref1 = ms_metric.compute(predictions=preds, references=[x[0] for x in ref1_only],
                                model_name_or_path="xlm-roberta-large", batch_size=4, device=device)
    ms_ref1_list = ms_ref1 if isinstance(ms_ref1, list) else ms_ref1.get("moverscore", ms_ref1)
    ms_results["ref1"] = float(np.nanmean(ms_ref1_list))

    # Ref2 (subset)
    if num_has_r2 > 0:
        ms_ref2 = ms_metric.compute(predictions=preds2, references=[x[0] for x in refs2],
                                    model_name_or_path="xlm-roberta-large", batch_size=4, device=device)
        ms_ref2_list = ms_ref2 if isinstance(ms_ref2, list) else ms_ref2.get("moverscore", ms_ref2)
        ms_results["ref2"] = float(np.nanmean(ms_ref2_list))
    else:
        ms_results["ref2"] = None

    # best-over-refs
    ncols = max((len(r) for r in refs_per_ex), default=1)
    per_ref_ms = []
    for i_ref in range(ncols):
        refs_i = [ (refs[i_ref] if i_ref < len(refs) else refs[0]) for refs in refs_per_ex ]
        ms = ms_metric.compute(predictions=preds, references=refs_i,
                               model_name_or_path="xlm-roberta-large", batch_size=4, device=device)
        per_ref_ms.append(ms if isinstance(ms, list) else ms.get("moverscore", ms))
    ms_best = []
    for j in range(len(records)):
        cand = [per_ref_ms[i_ref][j] for i_ref in range(len(per_ref_ms))]
        ms_best.append(np.nanmax(cand))
    ms_results["best"] = float(np.nanmean(ms_best))
except Exception as e:
    print("evaluate.moverscore failed, trying direct library:", e)
    try:
        from moverscore import word_mover_score
        # Ref1
        ms_ref1_list = []
        for j in tqdm(range(len(records)), desc="MoverScore ref1"):
            r = records[j]["references"][0] if records[j]["references"] else ""
            sc = word_mover_score([r], [preds[j]],
                                  model_name_or_path="xlm-roberta-large",
                                  stop_words=[], n_gram=1, remove_subwords=True, batch_size=4)
            ms_ref1_list.append(float(sc[0]))
        ms_results["ref1"] = float(np.nanmean(ms_ref1_list))
        # Ref2 subset
        if num_has_r2 > 0:
            ms_ref2_list = []
            for k, i in enumerate(tqdm([ix for ix in range(len(records)) if has_ref2_mask[ix]], desc="MoverScore ref2")):
                r = records[i]["references"][1]
                sc = word_mover_score([r], [preds[i]],
                                      model_name_or_path="xlm-roberta-large",
                                      stop_words=[], n_gram=1, remove_subwords=True, batch_size=4)
                ms_ref2_list.append(float(sc[0]))
            ms_results["ref2"] = float(np.nanmean(ms_ref2_list))
        else:
            ms_results["ref2"] = None
        # best-over-refs
        ms_best = []
        for j in tqdm(range(len(records)), desc="MoverScore best-over-refs"):
            refs = records[j]["references"] if records[j]["references"] else [""]
            cand = []
            for r in refs:
                sc = word_mover_score([r], [preds[j]],
                                      model_name_or_path="xlm-roberta-large",
                                      stop_words=[], n_gram=1, remove_subwords=True, batch_size=4)
                cand.append(float(sc[0]))
            ms_best.append(max(cand) if len(cand) else float("nan"))
        ms_results["best"] = float(np.nanmean(ms_best))
    except Exception as e2:
        print("MoverScore skipped:", e2)
        ms_results = {"ref1": None, "ref2": None, "best": None}

# =================== Summary ===================
metrics = {
    "count": len(records),

    # ROUGE (means)
    "rouge1_ref1_mean": mean_rouge(r1_rouge, "rouge1"),
    "rouge2_ref1_mean": mean_rouge(r1_rouge, "rouge2"),
    "rougeL_ref1_mean": mean_rouge(r1_rouge, "rougeL"),
    "rougeLsum_ref1_mean": mean_rouge(r1_rouge, "rougeLsum"),
    "rouge1_ref2_mean": mean_rouge(r2_rouge, "rouge1"),
    "rouge2_ref2_mean": mean_rouge(r2_rouge, "rouge2"),
    "rougeL_ref2_mean": mean_rouge(r2_rouge, "rougeL"),
    "rougeLsum_ref2_mean": mean_rouge(r2_rouge, "rougeLsum"),

    # BLEU / chrF
    "sacrebleu_multi_ref": sacrebleu.get("score", None),
    "sacrebleu_ref1": bleu_ref1,
    "sacrebleu_ref2": bleu_ref2,
    "chrf_multi_ref": chrf.get("score", None),
    "chrf_ref1": chrf_ref1,
    "chrf_ref2": chrf_ref2,

    # METEOR
    "meteor_ref1": meteor_ref1,
    "meteor_ref2": meteor_ref2,
    "meteor_best_over_refs": meteor_best,

    # BERTScore
    "bertscore_ref1": bertscore["ref1"],
    "bertscore_ref2": bertscore["ref2"],
    "bertscore_best_over_refs": bertscore["best"],

    # MoverScore
    "moverscore_ref1": ms_results["ref1"],
    "moverscore_ref2": ms_results["ref2"],
    "moverscore_best_over_refs": ms_results["best"],
}

print("\n=== Overall Metrics ===")
for k,v in metrics.items():
    print(f"{k}: {v}")

# =================== Save per-example table + summary ===================
rows = []
for i, r in enumerate(records):
    row = {
        "id": r["id"],
        "title": r["title"],
        "prediction": preds[i],
        "ref1": (r["references"][0] if len(r["references"])>=1 else ""),
        "ref2": (r["references"][1] if len(r["references"])>=2 else ""),
        "rouge1_ref1": r1_rouge[i]["rouge1"],
        "rouge2_ref1": r1_rouge[i]["rouge2"],
        "rougeL_ref1": r1_rouge[i]["rougeL"],
        "rougeLsum_ref1": r1_rouge[i]["rougeLsum"],
        "rouge1_ref2": r2_rouge[i]["rouge1"],
        "rouge2_ref2": r2_rouge[i]["rouge2"],
        "rougeL_ref2": r2_rouge[i]["rougeL"],
        "rougeLsum_ref2": r2_rouge[i]["rougeLsum"],
        "best_ref_used": (best_ref_choice[i] if i < len(best_ref_choice) else "")
    }
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, "test_preds_refs_metrics.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"\n📄 Saved rows  -> {csv_path}")
print(f"📊 Saved metrics -> {os.path.join(OUT_DIR, 'metrics_summary.json')}")


In [ ]:
# =================== One-Cell: Robust Summarize + Smart ReRank + Full Eval (Arabic) ===================
# Safe to run on Colab (A100/CPU). No empty generations; auto-retry with robust sampler when needed.
!pip -q install "transformers==4.44.2" "evaluate==0.4.2" "rouge_score==0.1.2" \
                "sacrebleu==2.4.0" "bert-score==0.3.13" "pot>=0.9.3" "pyemd>=1.0.0" \
                "nltk>=3.9" "tqdm"

import os, json, re, random, math, sys, time
from datetime import datetime
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel, GenerationConfig

# ---------- (اختياري) ربط Google Drive ----------
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# ---------- المسارات ----------
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"   # (id, title, Merged_Articles, target_summary, target_summary_2)
MODEL_DIR = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/final_bestx2"
OUT_ROOT  = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/test_eval_UNIFIED"
os.makedirs(OUT_ROOT, exist_ok=True)

# ---------- البيئة ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

MAX_INPUT = 6144
BERT_EMB  = "xlm-roberta-large"
BERT_BATCH= 8

# =================== Loader (مرن) ===================
def load_any_json_or_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s: continue
            if s.startswith("```"): continue
            if s.startswith("`"): s = s[1:]
            s = s.rstrip("`,")
            # cut to first JSON-like char
            if "{" in s or "[" in s:
                cuts = [i for i in (s.find("{"), s.find("[")) if i != -1]
                if cuts:
                    i0 = min(cuts)
                    if i0 > 0: s = s[i0:]
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj); continue
                elif isinstance(obj, list): recs.extend([x for x in obj if isinstance(x, dict)]); continue
            except json.JSONDecodeError:
                pass
    if recs: return recs
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        text = f.read().strip()
    if text.startswith("```"):
        parts = text.split("```")
        for blk in parts:
            if "{" in blk or "[" in blk:
                text = blk.strip(); break
    text = text.lstrip("`").rstrip("`")
    data = json.loads(text)
    if isinstance(data, dict): return [data]
    if isinstance(data, list): return [x for x in data if isinstance(x, dict)]
    raise ValueError("Unsupported JSON structure")

# =================== تهيئة عربية ===================
AR_PREFIX = "لخص بالعربية الفصحى بدقة واختصار ووضوح:"
_dup_open_re = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+')

def clean_merged(s: str) -> str:
    return _dup_open_re.sub(r'\1', s or "")

def ensure_prefix(s: str) -> str:
    s = s or ""
    return s if s.lstrip().startswith(AR_PREFIX) else (AR_PREFIX + " " + s)

def tidy_ar(s: str) -> str:
    if not s: return s
    s = re.sub(r"\s+", " ", s)
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s.strip()

def tokenize_ar_simple(s: str) -> List[str]:
    s = re.sub(r"[^\w\s%]+", " ", s, flags=re.UNICODE)
    toks = [t for t in s.split() if t]
    return toks

def extract_input_and_refs(rec: Dict[str, Any]):
    inp = rec.get("Merged_Articles", None)
    inp = ensure_prefix(clean_merged(inp)) if isinstance(inp, str) else None
    refs = []
    for k in ["target_summary", "target_summary_2"]:
        v = rec.get(k, None)
        if isinstance(v, str) and v.strip():
            refs.append(tidy_ar(v))
    return inp, refs

# =================== نموذج التلخيص ===================
tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
try:
    gen_cfg_base = GenerationConfig.from_pretrained(MODEL_DIR)
except Exception:
    gen_cfg_base = GenerationConfig()

# ضمان الرموز
pad_id = tok.pad_token_id
if pad_id is None:
    tok.add_special_tokens({'pad_token': '<pad>'})
    model.resize_token_embeddings(len(tok))
    pad_id = tok.pad_token_id
if getattr(model.config, "decoder_start_token_id", None) is None:
    model.config.decoder_start_token_id = pad_id
if getattr(gen_cfg_base, "decoder_start_token_id", None) is None:
    gen_cfg_base.decoder_start_token_id = pad_id
if getattr(model.config, "eos_token_id", None) is None and tok.eos_token_id is not None:
    model.config.eos_token_id = tok.eos_token_id

# =================== توليد + إعادة ترتيب (مع حماية من الفراغ) ===================
def est_target_len_words(src_text: str) -> int:
    nsrc = max(1, len(tokenize_ar_simple(src_text)))
    return int(np.clip(nsrc/18, 80, 200))

def _gen(enc_kwargs, gen_kwargs):
    with torch.no_grad():
        out = model.generate(**enc_kwargs, **gen_kwargs)
    seq = out.sequences[0]
    text_out = tok.decode(seq, skip_special_tokens=True)
    # avg logprob
    scores = out.scores
    gen_len = len(scores)
    if gen_len == 0:
        avg_ll = -1e9
    else:
        gen_tokens = seq[-gen_len:]
        ll = 0.0
        for t, logits in enumerate(scores):
            logp = torch.log_softmax(logits[0], dim=-1)
            ll += float(logp[gen_tokens[t]].item())
        avg_ll = ll / max(1, gen_len)
    return tidy_ar(text_out), avg_ll

@torch.no_grad()
def gen_once(text: str, strategy: str, target_words: int):
    no_repeat = 4
    rep_pen  = 1.2
    min_new  = max(80, int(target_words*0.6))
    max_new  = int(np.clip(int(target_words*1.6), 160, 512))
    min_new  = min(min_new, max_new-10)
    enc = tok([text], return_tensors="pt", truncation=True, max_length=MAX_INPUT, padding=True).to(device)
    base_kwargs = dict(
        return_dict_in_generate=True, output_scores=True,
        no_repeat_ngram_size=no_repeat, repetition_penalty=rep_pen,
        min_new_tokens=min_new, max_new_tokens=max_new, early_stopping=True
    )

    if strategy == "beam":
        cfg = gen_cfg_base.clone(); cfg.num_beams = 6; cfg.do_sample = False; cfg.length_penalty = 1.0
        out, ll = _gen(enc, {**base_kwargs, "generation_config": cfg})
    elif strategy == "nucleus":
        cfg = gen_cfg_base.clone(); cfg.do_sample = True; cfg.top_p = 0.92; cfg.temperature = 0.8; cfg.num_beams = 1
        out, ll = _gen(enc, {**base_kwargs, "generation_config": cfg})
    elif strategy == "beam_diverse":
        cfg = gen_cfg_base.clone(); cfg.num_beams = 6; cfg.num_beam_groups = 3; cfg.diversity_penalty = 0.2
        cfg.do_sample = True; cfg.top_k = 50; cfg.temperature = 0.9
        out, ll = _gen(enc, {**base_kwargs, "generation_config": cfg})
    elif strategy == "contrastive":
        # contrastive: keep the same max_new/min_new
        out, ll = _gen(enc, {**base_kwargs, "penalty_alpha": 0.6, "top_k": 8})
    else:
        raise ValueError("Unknown strategy")

    # إذا كان الناتج قصيرًا جدًا أو فارغًا، أعد المحاولة بوصفة آمنة
    if len(tokenize_ar_simple(out)) < 40:
        cfg = gen_cfg_base.clone()
        cfg.do_sample = True; cfg.top_p = 0.95; cfg.temperature = 0.9; cfg.num_beams = 1
        safe_kwargs = dict(
            return_dict_in_generate=True, output_scores=True,
            no_repeat_ngram_size=no_repeat, repetition_penalty=1.15,
            min_new_tokens=max(100, min_new), max_new_tokens=max(240, max_new), early_stopping=True,
            generation_config=cfg
        )
        out2, ll2 = _gen(enc, safe_kwargs)
        if len(tokenize_ar_simple(out2)) > len(tokenize_ar_simple(out)):
            return out2, ll2
    return out, ll

def bigram_redundancy(text: str) -> float:
    toks = tokenize_ar_simple(text);
    if len(toks) < 2: return 0.0
    bigs = [tuple(toks[i:i+2]) for i in range(len(toks)-1)]
    uniq = len(set(bigs))
    return 1.0 - (uniq / max(1, len(bigs)))  # 0=لا تكرار

def coverage_score(src: str, hyp: str) -> float:
    src_set = set(tokenize_ar_simple(src))
    if not src_set: return 0.0
    hyp_t = tokenize_ar_simple(hyp)
    if not hyp_t: return 0.0
    hit = sum(1 for w in hyp_t if w in src_set)
    return hit / len(hyp_t)

def length_fit_score(hyp: str, target_words: int) -> float:
    n = len(tokenize_ar_simple(hyp))
    return float(np.exp(-abs(n - target_words) / (0.6*target_words + 1e-6)))

def numeric_consistency(src: str, hyp: str) -> float:
    s_nums = set(re.findall(r"\d+[%]?", src))
    if not s_nums: return 1.0
    h_nums = set(re.findall(r"\d+[%]?", hyp))
    if not h_nums: return 0.6
    extra = len([x for x in h_nums if x not in s_nums])
    ok = len([x for x in h_nums if x in s_nums])
    return max(0.0, min(1.0, (ok + 1) / (ok + extra + 1)))

def composite_score(src: str, hyp: str, avg_ll: float, target_words: int) -> float:
    cov = coverage_score(src, hyp)
    flu = (avg_ll + 10) / 10         # rough norm
    lng = length_fit_score(hyp, target_words)
    red = bigram_redundancy(hyp)
    num = numeric_consistency(src, hyp)
    return 0.40*cov + 0.25*flu + 0.15*lng + 0.10*(1-red) + 0.10*num

def summarize_best(src_text: str):
    tw = est_target_len_words(src_text)
    cands = []
    for strat in ["beam", "nucleus", "contrastive", "beam_diverse"]:
        try:
            hyp, avg_ll = gen_once(src_text, strat, tw)
            sc = composite_score(src_text, hyp, avg_ll, tw)
            cands.append((sc, hyp, strat, avg_ll))
        except Exception:
            cands.append((-1e9, "", strat, -1e9))
    cands.sort(key=lambda x: x[0], reverse=True)
    return cands[0], cands

# =================== Generate ===================
raw = load_any_json_or_jsonl(TEST_PATH)
print(f"Loaded test items: {len(raw)}")

records = []
for i, rec in enumerate(raw):
    inp, refs = extract_input_and_refs(rec)
    records.append({
        "id": rec.get("id", i),
        "title": rec.get("title", ""),
        "input": inp,
        "references": refs,
        "prediction": None,
        "gen_info": {}
    })

for rec in tqdm(records, desc="Generating (multi-strategy + rerank)"):
    if not rec["input"]:
        rec["prediction"] = ""
        rec["gen_info"] = {"strategy": None, "avg_logprob": None, "score": None}
        continue
    (best, _all) = summarize_best(rec["input"])
    sc, hyp, strat, avg_ll = best
    # حماية إضافية: لا نسمح بفراغ نهائي
    if len(tokenize_ar_simple(hyp)) < 20:
        hyp = rec["input"][:400]  # fallback مستحيل يحصل، لكن كآخر حماية
    rec["prediction"] = tidy_ar(hyp)
    rec["gen_info"] = {"strategy": strat, "avg_logprob": avg_ll, "score": sc}

preds = [r["prediction"] or "" for r in records]

# =================== References ===================
refs_per_ex, ref1_only, has_ref2_mask = [], [], []
for r in records:
    refs = r["references"][:] if r["references"] else [""]
    refs_per_ex.append(refs)
    ref1_only.append([refs[0] if len(refs) >= 1 else ""])
    has_ref2_mask.append(bool(len(refs) >= 2 and (str(refs[1]).strip() != "")))
num_has_r2 = int(sum(has_ref2_mask))

if num_has_r2 > 0:
    idx2   = [i for i, ok in enumerate(has_ref2_mask) if ok]
    preds2 = [preds[i] for i in idx2]
    refs2  = [[records[i]["references"][1]] for i in idx2]
else:
    idx2, preds2, refs2 = [], [], []

# =================== ROUGE ===================
rouge_metric = evaluate.load("rouge")
def rouge_pair(pred, ref):
    sc = rouge_metric.compute(predictions=[pred], references=[ref], use_stemmer=False)
    return {k: float(sc[k]) for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

r1_rouge, r2_rouge, best_ref_choice = [], [], []
for i, r in tqdm(list(enumerate(records)), total=len(records), desc="Scoring (ROUGE)"):
    pred = preds[i]; refs = r["references"] if r["references"] else [""]
    sc1 = rouge_pair(pred, refs[0]); r1_rouge.append(sc1)
    if has_ref2_mask[i]:
        sc2 = rouge_pair(pred, refs[1]); r2_rouge.append(sc2)
        cand = [(sc1["rougeLsum"], refs[0]), (sc2["rougeLsum"], refs[1])]
    else:
        r2_rouge.append({"rouge1": np.nan, "rouge2": np.nan, "rougeL": np.nan, "rougeLsum": np.nan})
        cand = [(sc1["rougeLsum"], refs[0])]
    best_ref_choice.append(max(cand, key=lambda x: x[0])[1])

def mean_rouge(rows, key):
    arr = np.array([x[key] for x in rows], dtype=float)
    return float(np.nanmean(arr)) if np.any(~np.isnan(arr)) else None

# =================== BLEU / chrF ===================
sacrebleu_metric = evaluate.load("sacrebleu")
sacrebleu_multi = sacrebleu_metric.compute(predictions=preds, references=refs_per_ex) if refs_per_ex else {"score": None}
bleu_ref1 = sacrebleu_metric.compute(predictions=preds, references=[[x[0]] for x in ref1_only])["score"] if records else None
bleu_ref2 = (sacrebleu_metric.compute(predictions=preds2, references=refs2)["score"] if num_has_r2 > 0 else None)

try:
    chrf_metric = evaluate.load("chrf")
    chrf_multi = chrf_metric.compute(predictions=preds, references=refs_per_ex) if refs_per_ex else {"score": None}
    chrf_ref1  = chrf_metric.compute(predictions=preds, references=[[x[0]] for x in ref1_only])["score"] if records else None
    chrf_ref2  = (chrf_metric.compute(predictions=preds2, references=refs2)["score"] if num_has_r2 > 0 else None)
except Exception:
    chrf_multi = {"score": None}; chrf_ref1 = None; chrf_ref2 = None

# =================== METEOR ===================
try:
    meteor_metric = evaluate.load("meteor")
    meteor_ref1 = float(meteor_metric.compute(predictions=preds, references=[x[0] for x in ref1_only]).get("meteor", np.nan))
    meteor_ref2 = float(meteor_metric.compute(predictions=preds2, references=[x[0] for x in refs2]).get("meteor", np.nan)) if num_has_r2>0 else None
    meteor_best = float(meteor_metric.compute(predictions=preds, references=best_ref_choice).get("meteor", np.nan))
except Exception as e:
    print("METEOR skipped:", e)
    meteor_ref1 = meteor_ref2 = meteor_best = None

# =================== BERTScore (direct, IDF) ===================
from bert_score import BERTScorer
print("\n🎯 BERTScore (per-ref + best) — direct scorer")
all_refs_flat_for_idf = [r for rr in refs_per_ex for r in rr if str(r).strip()]
bs_device = "cuda" if torch.cuda.is_available() else "cpu"
bs_scorer = BERTScorer(lang="ar", model_type=BERT_EMB, batch_size=BERT_BATCH, idf=True, device=bs_device)
if len(all_refs_flat_for_idf) > 0:
    bs_scorer.compute_idf(all_refs_flat_for_idf)

def _score_bert(preds_list, refs_list):
    P, R, F = bs_scorer.score(preds_list, refs_list)
    return np.array(P.tolist()), np.array(R.tolist()), np.array(F.tolist())

P1, R1, F1 = _score_bert(preds, [x[0] for x in ref1_only])
r2_refs_fb = [(records[i]["references"][1] if has_ref2_mask[i] else (records[i]["references"][0] if records[i]["references"] else "")) for i in range(len(records))]
P2, R2, F2 = _score_bert(preds, r2_refs_fb)
mask_no_r2 = np.array([not x for x in has_ref2_mask])
P2[mask_no_r2] = np.nan; R2[mask_no_r2] = np.nan; F2[mask_no_r2] = np.nan

def ref_column(i_ref: int):
    return [(refs[i_ref] if i_ref < len(refs) else refs[0]) for refs in refs_per_ex]
ncols = max((len(r) for r in refs_per_ex), default=1)
per_ref_scores = []
for i_ref in range(ncols):
    Pi, Ri, Fi = _score_bert(preds, ref_column(i_ref))
    per_ref_scores.append((Pi, Ri, Fi))
P_best, R_best, F_best = [], [], []
for j in range(len(records)):
    fvals = [per_ref_scores[i][2][j] for i in range(ncols)]
    best_idx = int(np.nanargmax(fvals))
    P_best.append(per_ref_scores[best_idx][0][j])
    R_best.append(per_ref_scores[best_idx][1][j])
    F_best.append(per_ref_scores[best_idx][2][j])
P_best = np.array(P_best); R_best = np.array(R_best); F_best = np.array(F_best)

# =================== MoverScore (pyemd→POT) ===================
print("\n⚖️  MoverScore (per-ref + best)")
import ot
try:
    from pyemd import emd as pyemd_emd
    HAVE_PYEMD = True
except Exception as e:
    HAVE_PYEMD = False
    print("pyemd not available (likely NumPy 2.x ABI); using POT/ot.emd fallback.")

emb_tok = AutoTokenizer.from_pretrained(BERT_EMB)
emb_mdl = AutoModel.from_pretrained(BERT_EMB).to(device).eval()

@torch.no_grad()
def embed_tokens(text: str, max_len: int = 512) -> Tuple[torch.Tensor, List[int]]:
    out = emb_tok(text or "", return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=True)
    out = {k: v.to(device) for k, v in out.items()}
    hs = emb_mdl(**out).last_hidden_state.squeeze(0)
    ids = out["input_ids"].squeeze(0).tolist()
    special = set(t for t in [getattr(emb_tok, "cls_token_id", None), getattr(emb_tok, "sep_token_id", None), getattr(emb_tok, "pad_token_id", None)] if t is not None)
    keep = [i for i, tid in enumerate(ids) if tid not in special] or list(range(len(ids)))
    return hs[keep].detach().cpu(), [ids[i] for i in keep]

def build_idf_from_texts(texts: List[str], max_len: int = 512) -> dict:
    from collections import Counter, defaultdict
    df_counter = Counter()
    for t in texts:
        if not t: continue
        ids = emb_tok(t, truncation=True, max_length=max_len, add_special_tokens=True)["input_ids"]
        df_counter.update(set(ids))
    N = max(1, len([t for t in texts if t]))
    idf = defaultdict(lambda: 0.0)
    for tid, dfv in df_counter.items():
        idf[tid] = math.log((N + 1) / (dfv + 1))
    return idf

def cosine_cost(X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    Xn = torch.nn.functional.normalize(X, dim=1)
    Yn = torch.nn.functional.normalize(Y, dim=1)
    sim = Xn @ Yn.T
    return (1.0 - sim).clamp(min=0.0)

_EMB_CACHE = {}
def embed_cached(text: str):
    key = text if isinstance(text, str) else str(text)
    if key not in _EMB_CACHE:
        _EMB_CACHE[key] = embed_tokens(key, 512)
    return _EMB_CACHE[key]

idf_pred = build_idf_from_texts(preds)
all_refs_flat = [r for rr in refs_per_ex for r in rr]
idf_ref_flat = build_idf_from_texts(all_refs_flat)

def moverscore_pot(hyp: str, ref: str) -> float:
    H, ids_h = embed_cached(hyp); R, ids_r = embed_cached(ref)
    if H.shape[0] == 0 or R.shape[0] == 0: return 0.0
    a = torch.tensor([idf_pred.get(tid, 0.0) for tid in ids_h], dtype=torch.float32)
    b = torch.tensor([idf_ref_flat.get(tid, 0.0)  for tid in ids_r], dtype=torch.float32)
    if float(a.sum()) == 0: a = torch.ones_like(a)
    if float(b.sum()) == 0: b = torch.ones_like(b)
    a = (a / a.sum()).cpu().numpy(); b = (b / b.sum()).cpu().numpy()
    C = cosine_cost(H, R).cpu().numpy().astype(np.float64)
    T = ot.emd(a, b, C); cost = float((T * C).sum())
    return max(0.0, min(1.0, 1.0 - cost))

def moverscore_pyemd(hyp: str, ref: str) -> float:
    H, ids_h = embed_cached(hyp); R, ids_r = embed_cached(ref)
    if H.shape[0] == 0 or R.shape[0] == 0: return 0.0
    a = torch.tensor([idf_pred.get(tid, 0.0) for tid in ids_h], dtype=torch.float32)
    b = torch.tensor([idf_ref_flat.get(tid, 0.0)  for tid in ids_r], dtype=torch.float32)
    if float(a.sum()) == 0: a = torch.ones_like(a)
    if float(b.sum()) == 0: b = torch.ones_like(b)
    a = (a / a.sum()).cpu().numpy(); b = (b / b.sum()).cpu().numpy()
    C = cosine_cost(H, R).cpu().numpy().astype(np.float64)
    Th, Tr = C.shape; dim = Th + Tr
    CC = np.zeros((dim, dim), dtype=np.float64); CC[:Th, :Tr] = C
    aa = np.concatenate([a, np.zeros(Tr, dtype=np.float64)]); bb = np.concatenate([b, np.zeros(Th, dtype=np.float64)])
    s = max(aa.sum(), bb.sum())
    if s == 0: return 0.0
    aa /= s; bb /= s
    cost = pyemd_emd(aa, bb, CC)
    return max(0.0, min(1.0, 1.0 - float(cost)))

engine_fn = moverscore_pyemd if 'pyemd_emd' in globals() else moverscore_pot

def compute_ms_for_pairs(preds_list, refs_list):
    out = []
    for hyp, ref in tqdm(list(zip(preds_list, refs_list)), total=len(preds_list), desc="MoverScore"):
        out.append(engine_fn(hyp, ref))
    return np.array(out, dtype=float)

ms_ref1_arr = compute_ms_for_pairs(preds, [r["references"][0] if r["references"] else "" for r in records])
ms_ref1 = float(np.nanmean(ms_ref1_arr))
ms_ref2_arr = np.full(len(records), np.nan, dtype=float)
if num_has_r2 > 0:
    r2_refs, idxs = [], []
    for i in range(len(records)):
        if has_ref2_mask[i]:
            r2_refs.append(records[i]["references"][1]); idxs.append(i)
    ms_vals = compute_ms_for_pairs([preds[i] for i in idxs], r2_refs)
    for pos, i in enumerate(idxs): ms_ref2_arr[i] = ms_vals[pos]
ms_ref2 = (None if num_has_r2==0 else float(np.nanmean(ms_ref2_arr)))
ms_best_arr = []
for i in range(len(records)):
    refs = records[i]["references"] if records[i]["references"] else [""]
    cand = [engine_fn(preds[i], r) for r in refs]
    ms_best_arr.append(max(cand) if len(cand) else float("nan"))
ms_best = float(np.nanmean(ms_best_arr))

# =================== Summary ===================
def mean_rouge(rows, key):
    arr = np.array([x[key] for x in rows], dtype=float)
    return float(np.nanmean(arr)) if np.any(~np.isnan(arr)) else None

metrics = {
    "count": len(records),

    # ROUGE
    "rouge1_ref1_mean": mean_rouge(r1_rouge, "rouge1"),
    "rouge2_ref1_mean": mean_rouge(r1_rouge, "rouge2"),
    "rougeL_ref1_mean": mean_rouge(r1_rouge, "rougeL"),
    "rougeLsum_ref1_mean": mean_rouge(r1_rouge, "rougeLsum"),
    "rouge1_ref2_mean": mean_rouge(r2_rouge, "rouge1"),
    "rouge2_ref2_mean": mean_rouge(r2_rouge, "rouge2"),
    "rougeL_ref2_mean": mean_rouge(r2_rouge, "rougeL"),
    "rougeLsum_ref2_mean": mean_rouge(r2_rouge, "rougeLsum"),

    # BLEU / chrF
    "sacrebleu_multi_ref": sacrebleu_metric.compute(predictions=preds, references=refs_per_ex)["score"] if refs_per_ex else None,
    "sacrebleu_ref1": sacrebleu_metric.compute(predictions=preds, references=[[x[0]] for x in ref1_only])["score"] if records else None,
    "sacrebleu_ref2": sacrebleu_metric.compute(predictions=preds2, references=refs2)["score"] if num_has_r2 > 0 else None,
    "chrf_multi_ref": (evaluate.load("chrf").compute(predictions=preds, references=refs_per_ex)["score"] if refs_per_ex else None),
    "chrf_ref1": (evaluate.load("chrf").compute(predictions=preds, references=[[x[0]] for x in ref1_only])["score"] if records else None),
    "chrf_ref2": (evaluate.load("chrf").compute(predictions=preds2, references=refs2)["score"] if num_has_r2 > 0 else None),

    # METEOR
    "meteor_ref1": (evaluate.load("meteor").compute(predictions=preds, references=[x[0] for x in ref1_only]).get("meteor") if records else None),
    "meteor_ref2": (evaluate.load("meteor").compute(predictions=preds2, references=[x[0] for x in refs2]).get("meteor") if num_has_r2>0 else None),
    "meteor_best_over_refs": (evaluate.load("meteor").compute(predictions=preds, references=best_ref_choice).get("meteor") if records else None),

    # BERTScore
    "bertscore_ref1": {"P": float(np.nanmean(P1)), "R": float(np.nanmean(R1)), "F1": float(np.nanmean(F1))},
    "bertscore_ref2": (None if num_has_r2==0 else {"P": float(np.nanmean(P2)), "R": float(np.nanmean(R2)), "F1": float(np.nanmean(F2))}),
    "bertscore_best_over_refs": {"P": float(np.nanmean(P_best)), "R": float(np.nanmean(R_best)), "F1": float(np.nanmean(F_best))},

    # MoverScore
    "moverscore_ref1": ms_ref1,
    "moverscore_ref2": ms_ref2,
    "moverscore_best_over_refs": ms_best,
}

print("\n=== Overall Metrics (Unified, robust) ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

# =================== Save ===================
rows = []
for i, r in enumerate(records):
    row = {
        "id": r["id"],
        "title": r["title"],
        "strategy": r["gen_info"].get("strategy"),
        "avg_logprob": r["gen_info"].get("avg_logprob"),
        "rerank_score": r["gen_info"].get("score"),
        "prediction": preds[i],
        "ref1": (r["references"][0] if len(r["references"])>=1 else ""),
        "ref2": (r["references"][1] if len(r["references"])>=2 else ""),
        "rouge1_ref1": r1_rouge[i]["rouge1"],
        "rouge2_ref1": r1_rouge[i]["rouge2"],
        "rougeL_ref1": r1_rouge[i]["rougeL"],
        "rougeLsum_ref1": r1_rouge[i]["rougeLsum"],
        "rouge1_ref2": r2_rouge[i]["rouge1"],
        "rouge2_ref2": r2_rouge[i]["rouge2"],
        "rougeL_ref2": r2_rouge[i]["rougeL"],
        "rougeLsum_ref2": r2_rouge[i]["rougeLsum"],
    }
    rows.append(row)

df = pd.DataFrame(rows)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = os.path.join(OUT_ROOT, f"run_{ts}")
os.makedirs(OUT_DIR, exist_ok=True)

csv_path = os.path.join(OUT_DIR, "test_preds_refs_metrics.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"\n📄 Saved rows  -> {csv_path}")
print(f"📊 Saved metrics -> {os.path.join(OUT_DIR, 'metrics_summary.json')}")


In [ ]:
# ==== Print Top-3 Summaries with IDs & References (from memory or latest CSV) ====
import os, glob, pandas as pd
from typing import List, Dict, Any

TOP_N = 3
OUT_ROOT = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/test_eval_UNIFIED"  # adjust if needed

def _score_rec(rec: Dict[str, Any]) -> float:
    sc = (rec.get("gen_info") or {}).get("score")
    try:
        return float(sc)
    except Exception:
        return float(len(str(rec.get("prediction", ""))))  # fallback: length

def _print_one(idx:int, rec:Dict[str,Any], score:float):
    refs = rec.get("references") or []
    print(f"\n{'='*100}")
    print(f"#{idx} | ID: {rec.get('id')} | Strategy: {(rec.get('gen_info') or {}).get('strategy')} | Score: {score:.4f}")
    title = str(rec.get("title",""))
    if title: print(f"Title: {title[:120]}")
    print("\nPrediction:\n", rec.get("prediction",""))
    if len(refs)>=1 and isinstance(refs[0], str) and refs[0].strip():
        print("\nRef1:\n", refs[0])
    if len(refs)>=2 and isinstance(refs[1], str) and refs[1].strip():
        print("\nRef2:\n", refs[1])

def top_n_from_records(records: List[Dict[str, Any]], n:int=3):
    scored = [(_score_rec(r), r) for r in records]
    scored.sort(key=lambda x: x[0], reverse=True)
    for i, (sc, rec) in enumerate(scored[:n], 1):
        _print_one(i, rec, sc)

def top_n_from_csv(out_root:str, n:int=3):
    runs = sorted(glob.glob(os.path.join(out_root, "run_*")), key=os.path.getmtime, reverse=True)
    for rd in runs:
        csv_path = os.path.join(rd, "test_preds_refs_metrics.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            # use rerank_score if available else prediction length
            if "rerank_score" in df.columns:
                df["__score__"] = pd.to_numeric(df["rerank_score"], errors="coerce")
            else:
                df["__score__"] = None
            df["__score__"] = df["__score__"].fillna(df["prediction"].astype(str).str.len().astype(float))
            df = df.sort_values("__score__", ascending=False).head(n)
            for i, row in enumerate(df.itertuples(index=False), 1):
                rec = {
                    "id": getattr(row, "id", None),
                    "title": getattr(row, "title", ""),
                    "prediction": getattr(row, "prediction", ""),
                    "references": [getattr(row, "ref1", "") , getattr(row, "ref2", "")],
                    "gen_info": {"strategy": getattr(row, "strategy", None)}
                }
                _print_one(i, rec, float(getattr(row, "__score__", 0.0)))
            print(f"\n(From {csv_path})")
            return
    print("No CSV found under OUT_ROOT. Set OUT_ROOT correctly or run generation first.")

# --- Use in-memory 'records' if available; else load latest CSV ---
try:
    records  # will raise NameError if not defined
    print("Using in-memory 'records'")
    top_n_from_records(records, TOP_N)
except NameError:
    print("No in-memory 'records' found; scanning OUT_ROOT for latest run…")
    top_n_from_csv(OUT_ROOT, TOP_N)


In [ ]:
# =================== Stronger Arabic Summarization (length-aware + multi-candidate + MBR rerank) ===================
!pip -q install "transformers==4.44.2" "evaluate==0.4.2" "rouge_score==0.1.2" "tqdm"

import os, json, re, random, math, sys, copy
from datetime import datetime
from typing import List, Dict, Any, Tuple

import numpy as np
import torch
from tqdm.auto import tqdm
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig

# --------- (اختياري) ربط Google Drive ---------
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# --------- Paths ---------
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
MODEL_DIR = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/final_bestx2"
OUT_DIR   = f"/content/drive/MyDrive/mt5_xlsum444_patch_6144/test_eval_ENHANCED_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

os.makedirs(OUT_DIR, exist_ok=True)

# --------- Env ---------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

MAX_INPUT_TOKS   = 4096   # حد إدخال عملي
TARGET_WORDS = (250, 320)  # نطاق الكلمات المطلوب
CAND_K           = 6           # عدد المرشحات قبل MBR
LONG_THRESH_TOKS = 2200        # التلخيص الهرمي لما هو طويل جدًا

# =================== تنظيف المدخل/المخرج ===================
CTRL_PREFIX_RE = re.compile(
    r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*",
    flags=re.IGNORECASE
)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", flags=re.IGNORECASE)

def sanitize_for_prompt(src: str) -> str:
    s = src or ""
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def sanitize_output(summary: str) -> str:
    s = summary or ""
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# =================== Utils ===================
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    out = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("```"): continue
            if s.startswith("`"): s = s[1:]
            s = s.rstrip("`,")
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): out.append(obj)
            except json.JSONDecodeError:
                pass
    return out

_dup_open_re = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+')

def clean_merged(s: str) -> str:
    return _dup_open_re.sub(r'\1', s or "")

def tidy_ar(s: str) -> str:
    if not s: return s
    s = re.sub(r"\s+", " ", s)
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s.strip()

def tokenize_words(s: str) -> List[str]:
    s = re.sub(r"[^\w\s%]+", " ", s, flags=re.UNICODE)
    return [t for t in s.split() if t]

def word_count(s: str) -> int:
    return len(tokenize_words(s or ""))

def build_prompt(src: str, wmin: int, wmax: int) -> str:
    # Prompt بطول محدد + تغطية أرقام/أسباب دون نقاط/عناوين
    return (
        f"اكتب خلاصة عربية فصحى محايدة ومترابطة في {wmin}–{wmax} كلمة،"
        f" تُغطي الحقائق الأساسية والأرقام والأسباب والنتائج، دون تعداد نقطي أو عناوين:\n{src}"
    )

def split_by_articles(raw: str) -> List[str]:
    parts = re.split(r"(?:</?ARTICLE_\d+>)", raw or "", flags=re.IGNORECASE)
    chunks = [tidy_ar(p) for p in parts if p and not re.match(r"</?ARTICLE_\d+>", p, flags=re.IGNORECASE)]
    chunks = [c for c in chunks if c.strip()]
    return chunks if len(chunks) >= 2 else [raw]

# =================== Model ===================
tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
try:
    base_cfg = GenerationConfig.from_pretrained(MODEL_DIR)
except Exception:
    base_cfg = GenerationConfig()

# ضمان الرموز
pad_id = tok.pad_token_id
if pad_id is None:
    tok.add_special_tokens({'pad_token': '<pad>'})
    model.resize_token_embeddings(len(tok))
    pad_id = tok.pad_token_id
if getattr(model.config, "decoder_start_token_id", None) is None:
    model.config.decoder_start_token_id = pad_id
if getattr(base_cfg, "decoder_start_token_id", None) is None:
    base_cfg.decoder_start_token_id = pad_id
if getattr(model.config, "eos_token_id", None) is None and tok.eos_token_id is not None:
    model.config.eos_token_id = tok.eos_token_id

def make_cfg(**overrides) -> GenerationConfig:
    cfg = copy.deepcopy(base_cfg)
    for k, v in overrides.items():
        setattr(cfg, k, v)
    return cfg

# =================== Decoding strategies ===================
def _gen(enc, **kwargs) -> Tuple[str, float]:
    try:
        with torch.no_grad():
            out = model.generate(**enc, **kwargs, return_dict_in_generate=True, output_scores=True)
    except (TypeError, ValueError):
        # فالـباك عام: nucleus sampling
        fallback_cfg = make_cfg(do_sample=True, top_p=0.9, temperature=0.95)
        with torch.no_grad():
            out = model.generate(**enc, generation_config=fallback_cfg,
                                 no_repeat_ngram_size=kwargs.get("no_repeat_ngram_size", 4),
                                 repetition_penalty=kwargs.get("repetition_penalty", 1.08),
                                 min_new_tokens=kwargs.get("min_new_tokens", 160),
                                 max_new_tokens=kwargs.get("max_new_tokens", 320),
                                 early_stopping=True,
                                 return_dict_in_generate=True, output_scores=True)
    seq = out.sequences[0]
    text = tidy_ar(tok.decode(seq, skip_special_tokens=True))
    scores = out.scores or []
    if not scores:
        return text, -1e9
    gen_len = len(scores)
    gen_tokens = seq[-gen_len:]
    ll = 0.0
    for t, logits in enumerate(scores):
        logp = torch.log_softmax(logits[0], dim=-1)
        ll += float(logp[gen_tokens[t]].item())
    return text, ll / max(1, gen_len)

def target_to_tokens(wmin: int, wmax: int) -> Tuple[int, int]:
    # تقريب: عدد التوكنات ≈ 1.6 × الكلمات (تقريبي لـ mT5)
    min_new = int(max(120, wmin * 1.2))
    max_new = int(min(640, wmax * 1.8))
    if max_new - min_new < 40: max_new = min_new + 40
    return min_new, max_new

def gen_candidates(prompt: str, k: int, wmin: int, wmax: int) -> List[Tuple[str, float, str]]:
    min_new, max_new = target_to_tokens(wmin, wmax)
    enc = tok([prompt], return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKS, padding=True).to(device)
    base_kwargs = dict(no_repeat_ngram_size=4, repetition_penalty=1.08,
                       min_new_tokens=min_new, max_new_tokens=max_new, early_stopping=True)

    # 👇 أهم تعديل: diverse_beam الآن بدون sampling (do_sample=False) وفق قواعد HF
    configs = [
        ("beam",          dict(generation_config=make_cfg(num_beams=6, do_sample=False, length_penalty=1.2))),
        ("diverse_beam",  dict(generation_config=make_cfg(num_beams=8, num_beam_groups=4,
                                                          diversity_penalty=0.6, do_sample=False,
                                                          length_penalty=1.05))),
        ("nucleus",       dict(generation_config=make_cfg(do_sample=True, top_p=0.92, temperature=0.85))),
        ("temp_1",        dict(generation_config=make_cfg(do_sample=True, top_k=50, temperature=0.95))),
        ("temp_2",        dict(generation_config=make_cfg(do_sample=True, top_p=0.88, temperature=1.05))),
        ("contrastive",   dict(penalty_alpha=0.6, top_k=8)),  # قد لا تدعمها بعض إصدارات HF → نfallback تلقائيًا
    ]

    cands = []
    for name, kw in configs[:max(1, min(k, len(configs)))]:
        try:
            text, ll = _gen(enc, **base_kwargs, **kw)
        except Exception:
            # fallback إضافي لأي استثناء غير متوقَّع
            cfg = make_cfg(do_sample=True, top_p=0.9, temperature=0.95)
            text, ll = _gen(enc, **base_kwargs, generation_config=cfg)
            name = "fallback"
        # مانع القِصر
        if word_count(text) < int(wmin * 0.8):
            cfg = make_cfg(do_sample=True, top_p=0.96, temperature=1.0)
            t2, ll2 = _gen(enc, **dict(base_kwargs, generation_config=cfg,
                                       min_new_tokens=max(min_new, 160), max_new_tokens=max(max_new, 320),
                                       repetition_penalty=1.03))
            if word_count(t2) > word_count(text):
                text, ll, name = t2, ll2, f"{name}+retry"
        cands.append((sanitize_output(text), ll, name))
    return cands

# =================== MBR rerank (ROUGE-Lsum consensus) ===================
rouge_metric = evaluate.load("rouge")

def rouge_lsum(a: str, b: str) -> float:
    try:
        return float(rouge_metric.compute(predictions=[a], references=[b], use_stemmer=False)["rougeLsum"])
    except Exception:
        return 0.0

def mbr_pick(cands: List[str]) -> Tuple[int, List[float]]:
    n = len(cands)
    if n == 1: return 0, [1.0]
    util = np.zeros(n, dtype=float)
    for i in range(n):
        scs = []
        for j in range(n):
            if i == j: continue
            scs.append(rouge_lsum(cands[i], cands[j]))
        util[i] = float(np.mean(scs)) if scs else 0.0
    return int(np.argmax(util)), util.tolist()

# =================== Hierarchical summarization ===================
def hierarchical_summarize(raw_src: str, wmin: int, wmax: int) -> str:
    parts = split_by_articles(raw_src)
    sub_wmin = max(90, int(wmin * 0.55))
    sub_wmax = max(120, int(wmax * 0.65))
    sub_summaries = []
    for p in parts:
        p_clean = sanitize_for_prompt(p)
        prompt = build_prompt(p_clean, sub_wmin, sub_wmax)
        cands = gen_candidates(prompt, k=min(4, CAND_K), wmin=sub_wmin, wmax=sub_wmax)
        best_idx, _ = mbr_pick([c[0] for c in cands])
        sub_summaries.append(cands[best_idx][0])
    merged = " ".join(sub_summaries)
    prompt_final = build_prompt(merged, wmin, wmax)
    cands = gen_candidates(prompt_final, k=CAND_K, wmin=wmin, wmax=wmax)
    best_idx, _ = mbr_pick([c[0] for c in cands])
    return cands[best_idx][0]

# =================== Run on TEST_PATH ===================
raw = load_jsonl(TEST_PATH)
print(f"Loaded {len(raw)} items")

records = []
for i, rec in enumerate(tqdm(raw, desc="Generating")):
    src_raw = clean_merged(rec.get("Merged_Articles", "") or "")
    src_clean = sanitize_for_prompt(src_raw)

    # قرّر المسار حسب طول المُدخل
    enc_len = len(tok(src_clean, truncation=False)["input_ids"])
    if enc_len > LONG_THRESH_TOKS:
        pred = hierarchical_summarize(src_clean, TARGET_WORDS[0], TARGET_WORDS[1])
        strat = "hierarchical+MBR"
    else:
        prompt = build_prompt(src_clean, TARGET_WORDS[0], TARGET_WORDS[1])
        cands = gen_candidates(prompt, k=CAND_K, wmin=TARGET_WORDS[0], wmax=TARGET_WORDS[1])
        best_idx, utils = mbr_pick([c[0] for c in cands])
        pred = cands[best_idx][0]
        strat = f"MBR({cands[best_idx][2]})"

    # حارس أخير ضد القِصر
    if word_count(pred) < TARGET_WORDS[0]*0.8:
        prompt2 = build_prompt(src_clean, TARGET_WORDS[0], TARGET_WORDS[1])
        more = gen_candidates(prompt2, k=max(3, CAND_K//2), wmin=TARGET_WORDS[0], wmax=TARGET_WORDS[1])
        best2, _ = mbr_pick([m[0] for m in more])
        if word_count(more[best2][0]) > word_count(pred):
            pred = more[best2][0]
            strat += "+final_retry"

    refs = []
    for k in ["target_summary","target_summary_2"]:
        v = rec.get(k, None)
        if isinstance(v, str) and v.strip():
            refs.append(tidy_ar(v))

    records.append({
        "id": rec.get("id", i),
        "title": rec.get("title", ""),
        "prediction": sanitize_output(pred),
        "references": refs,
        "strategy": strat
    })

# Save quick view
import pandas as pd, json as _json
df = pd.DataFrame([{
    "id": r["id"],
    "title": r["title"],
    "strategy": r["strategy"],
    "prediction": r["prediction"],
    "ref1": (r["references"][0] if len(r["references"])>=1 else ""),
    "ref2": (r["references"][1] if len(r["references"])>=2 else "")
} for r in records])

csv_path = os.path.join(OUT_DIR, "enhanced_predictions.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "enhanced_records.json"), "w", encoding="utf-8") as f:
    _json.dump(records, f, ensure_ascii=False, indent=2)

print(f"\n✅ Done. Saved:\n- {csv_path}\n- {os.path.join(OUT_DIR, 'enhanced_records.json')}\n")
# Quick sanity: show lengths
lens = [word_count(r["prediction"]) for r in records]
print(f"Length (words) — min:{min(lens)} | mean:{int(np.mean(lens))} | max:{max(lens)}")
print("Sample:\n", records[0]["prediction"][:600])


In [ ]:
# =================== Unified Evaluation (Ref1 / Ref2 / Avg / Best) — FIXED BERTScore IDF ===================
!pip -q install "pandas>=2.0" "evaluate==0.4.2" "rouge_score==0.1.2" "sacrebleu==2.4.0" \
                 "bert-score==0.3.13" "transformers>=4.41.0" "pot>=0.9.3" tqdm

import os, re, json, math
from typing import List, Dict, Any, Tuple
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel
import evaluate
from bert_score import BERTScorer
import ot  # POT

# --------- Paths ---------
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
PRED_PATH = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/test_eval_ENHANCED_20250924_231254/enhanced_predictions.csv"
OUT_DIR   = f"/content/drive/MyDrive/eval_runs/run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

# --------- Env ---------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ Device:", device)

# --------- Helpers ---------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)

def sanitize_text(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith("```"): continue
            if s.startswith("`"): s = s[1:]
            s = s.rstrip("`,")
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj)
            except json.JSONDecodeError:
                pass
    return recs

# --------- Load test + refs ---------
items = load_jsonl(TEST_PATH)
assert items, f"❌ لا يوجد عناصر في {TEST_PATH}"
print("🧪 Loaded items:", len(items))

ref1 = [sanitize_text(x.get("target_summary", "")) for x in items]
ref2 = [sanitize_text(x.get("target_summary_2", "")) for x in items]
has_r2 = [bool(r.strip()) for r in ref2]
num_r2 = sum(1 for x in has_r2 if x)
print(f"Ref1 count: {len(ref1)} | Ref2 non-empty: {num_r2}/{len(ref2)}")

# --------- Load preds ---------
def load_predictions(pred_path: str, test_items: List[Dict[str, Any]]) -> List[str]:
    id2pred = {}
    preds_list = []

    if pred_path.lower().endswith(".csv"):
        df = pd.read_csv(pred_path)
        pred_col = None
        for c in ["prediction", "pred", "output", "summary", "generated", "generated_summary"]:
            if c in df.columns: pred_col = c; break
        if pred_col is None:
            raise ValueError(f"لم أجد عمود التنبؤات في CSV. الأعمدة: {list(df.columns)}")
        if "id" in df.columns:
            for _, row in df.iterrows():
                id2pred[str(row["id"])] = sanitize_text(row[pred_col])
        else:
            preds_list = [sanitize_text(x) for x in df[pred_col].astype(str).tolist()]
    else:
        try:
            with open(pred_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict) and "records" in data: data = data["records"]
            if not isinstance(data, list): data = [data]
        except Exception:
            data = load_jsonl(pred_path)
        for obj in data:
            pred_txt = obj.get("prediction") or obj.get("pred") or obj.get("summary") or ""
            pred_txt = sanitize_text(pred_txt)
            if "id" in obj: id2pred[str(obj["id"])] = pred_txt
            else: preds_list.append(pred_txt)

    preds = []
    for i, rec in enumerate(test_items):
        key = str(rec.get("id", i))
        if id2pred:
            preds.append(id2pred.get(key, ""))
        else:
            preds.append(preds_list[i] if i < len(preds_list) else "")
    return preds

preds = load_predictions(PRED_PATH, items)
preds = [sanitize_text(p) for p in preds]
print("✅ Predictions loaded:", len(preds))

# --------- Build refs structures ---------
refs_multi = []
for i in range(len(items)):
    rlist = []
    if ref1[i].strip(): rlist.append(ref1[i])
    if ref2[i].strip(): rlist.append(ref2[i])
    if not rlist: rlist = [""]
    refs_multi.append(rlist)

# =================== ROUGE ===================
rouge = evaluate.load("rouge")
def rouge_scores(pred, ref):
    sc = rouge.compute(predictions=pred, references=ref, use_stemmer=False)
    return {k: float(sc[k]) for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

rouge_ref1 = rouge_scores(preds, ref1)
rouge_ref2 = rouge_scores(preds, [r if r else "" for r in ref2]) if num_r2>0 else {k: None for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

# best-over-refs (ROUGE-Lsum per-sample)
rouge1_list = []; rouge2_list = []; rougeL_list = []; rougeLsum_list = []; best_ref_ix=[]
for i in tqdm(range(len(items)), desc="ROUGE best-over-refs"):
    rs = [x for x in [ref1[i], ref2[i]] if x]
    if not rs: rs = [""]
    best = {"r1":0.0,"r2":0.0,"rL":0.0,"rLsum":0.0}; bix=0
    for j, r in enumerate(rs):
        sc = rouge.compute(predictions=[preds[i]], references=[r], use_stemmer=False)
        if sc["rougeLsum"] > best["rLsum"]:
            best = {"r1":float(sc["rouge1"]), "r2":float(sc["rouge2"]), "rL":float(sc["rougeL"]), "rLsum":float(sc["rougeLsum"])}
            bix=j
    rouge1_list.append(best["r1"]); rouge2_list.append(best["r2"])
    rougeL_list.append(best["rL"]); rougeLsum_list.append(best["rLsum"])
    best_ref_ix.append(bix)
rouge_best = {
    "rouge1": float(np.mean(rouge1_list)),
    "rouge2": float(np.mean(rouge2_list)),
    "rougeL": float(np.mean(rougeL_list)),
    "rougeLsum": float(np.mean(rougeLsum_list)),
}
rouge_avg = {k: None if rouge_ref2[k] is None else (rouge_ref1[k] + rouge_ref2[k]) / 2.0 for k in rouge_ref1}

# =================== SacreBLEU / chrF ===================
sacrebleu = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def wrap_single(refs):  # [[ref_i] ...]
    return [[r] for r in refs]

bleu_multi = sacrebleu.compute(predictions=preds, references=refs_multi)["score"]
bleu_r1    = sacrebleu.compute(predictions=preds, references=wrap_single(ref1))["score"]
bleu_r2    = sacrebleu.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
bleu_avg   = None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0

chrf_multi = chrf_metric.compute(predictions=preds, references=refs_multi)["score"]
chrf_r1    = chrf_metric.compute(predictions=preds, references=wrap_single(ref1))["score"]
chrf_r2    = chrf_metric.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
chrf_avg   = None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0

# =================== METEOR ===================
try:
    meteor_metric = evaluate.load("meteor")
    met_r1 = meteor_metric.compute(predictions=preds, references=ref1)["meteor"]
    met_r2 = meteor_metric.compute(predictions=preds, references=[r if r else "" for r in ref2])["meteor"] if num_r2>0 else None
    # best-over-refs
    met_best_list = []
    for i in tqdm(range(len(items)), desc="METEOR best-over-refs"):
        rs = [x for x in [ref1[i], ref2[i]] if x] or [""]
        best = 0.0
        for r in rs:
            best = max(best, meteor_metric.compute(predictions=[preds[i]], references=[r])["meteor"])
        met_best_list.append(best)
    met_best = float(np.mean(met_best_list))
except Exception as e:
    print("METEOR skipped:", e)
    met_r1 = met_r2 = met_best = None

# =================== BERTScore (direct scorer; FIX: compute_idf before score) ===================
print("\n🎯 BERTScore (per-ref + best) — direct scorer")
BERT_MODEL = "xlm-roberta-large"
scorer = BERTScorer(model_type=BERT_MODEL, lang=None, rescale_with_baseline=False, idf=True, device=device, batch_size=8)

# ✅ FIX: احسب IDF من كل المراجع المتاحة
all_refs_flat = [r for rs in refs_multi for r in rs]
scorer.compute_idf(all_refs_flat)

def bert_over(preds, refs):
    P,R,F = scorer.score(preds, refs)
    return float(torch.mean(P).item()), float(torch.mean(R).item()), float(torch.mean(F).item())

bP1,bR1,bF1 = bert_over(preds, ref1)
if num_r2>0:
    bP2,bR2,bF2 = bert_over(preds, [r if r else "" for r in ref2])
else:
    bP2=bR2=bF2=None

P1,R1,F1 = scorer.score(preds, ref1)
if num_r2>0:
    P2,R2,F2 = scorer.score(preds, [r if r else "" for r in ref2])
bP_best=[]; bR_best=[]; bF_best=[]
for i in range(len(items)):
    if num_r2>0 and F2[i] > F1[i]:
        bP_best.append(float(P2[i])); bR_best.append(float(R2[i])); bF_best.append(float(F2[i]))
    else:
        bP_best.append(float(P1[i])); bR_best.append(float(R1[i])); bF_best.append(float(F1[i]))
bP_best = float(np.mean(bP_best)); bR_best = float(np.mean(bR_best)); bF_best = float(np.mean(bF_best))

# =================== MoverScore (POT/OT) ===================
print("\n⚖️  MoverScore (per-ref + best) — POT/OT")
EMB_MODEL = "xlm-roberta-large"
MAX_LEN   = 512
tok_emb = AutoTokenizer.from_pretrained(EMB_MODEL)
mdl_emb = AutoModel.from_pretrained(EMB_MODEL).to(device).eval()

@torch.no_grad()
def embed_tokens(text: str, max_len: int = MAX_LEN) -> Tuple[torch.Tensor, List[int]]:
    out = tok_emb(text or "", return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=True)
    out = {k: v.to(device) for k, v in out.items()}
    hs = mdl_emb(**out).last_hidden_state.squeeze(0)
    ids = out["input_ids"].squeeze(0).tolist()
    special = set(t for t in [getattr(tok_emb, "cls_token_id", None),
                              getattr(tok_emb, "sep_token_id", None),
                              getattr(tok_emb, "pad_token_id", None)] if t is not None)
    keep = [i for i, tid in enumerate(ids) if tid not in special] or list(range(len(ids)))
    return hs[keep], [ids[i] for i in keep]

def build_idf_from_texts(texts: List[str], max_len: int = MAX_LEN) -> dict:
    from collections import Counter, defaultdict
    df_counter = Counter()
    for t in texts:
        ids = tok_emb(t or "", truncation=True, max_length=max_len, add_special_tokens=True)["input_ids"]
        df_counter.update(set(ids))
    N = max(1, len(texts))
    idf = defaultdict(lambda: 0.0)
    for tid, dfv in df_counter.items():
        idf[tid] = math.log((N + 1) / (dfv + 1))
    return idf

def cosine_cost(X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    Xn = torch.nn.functional.normalize(X, dim=1)
    Yn = torch.nn.functional.normalize(Y, dim=1)
    sim = Xn @ Yn.T
    return (1.0 - sim).clamp(min=0.0)

idf_pred = build_idf_from_texts(preds)
idf_ref1 = build_idf_from_texts(ref1)
idf_ref2 = build_idf_from_texts([r if r else "" for r in ref2]) if num_r2>0 else {}

def moverscore_pair(hyp: str, ref: str, idf_h: dict, idf_r: dict) -> float:
    H, ids_h = embed_tokens(hyp, MAX_LEN)
    R, ids_r = embed_tokens(ref, MAX_LEN)
    if H.shape[0] == 0 or R.shape[0] == 0: return 0.0
    a = torch.tensor([idf_h.get(tid, 0.0) for tid in ids_h], dtype=torch.float32)
    b = torch.tensor([idf_r.get(tid, 0.0) for tid in ids_r], dtype=torch.float32)
    if float(a.sum()) == 0: a = torch.ones_like(a)
    if float(b.sum()) == 0: b = torch.ones_like(b)
    a = (a / a.sum()).cpu().numpy()
    b = (b / b.sum()).cpu().numpy()
    C = cosine_cost(H, R).cpu().numpy().astype(np.float64)
    T = ot.emd(a, b, C)
    cost = float((T * C).sum())
    return max(0.0, min(1.0, 1.0 - cost))

ms_r1, ms_r2, ms_best = [], [], []
for i in tqdm(range(len(items)), desc="MoverScore"):
    p = preds[i]
    r1 = ref1[i] if ref1[i] else ""
    s1 = moverscore_pair(p, r1, idf_pred, idf_ref1)
    ms_r1.append(s1)
    if num_r2>0 and ref2[i]:
        s2 = moverscore_pair(p, ref2[i], idf_pred, idf_ref2)
        ms_r2.append(s2)
        ms_best.append(max(s1, s2))
    else:
        ms_best.append(s1)

mover_ref1 = float(np.mean(ms_r1))
mover_ref2 = float(np.mean(ms_r2)) if num_r2>0 and ms_r2 else None
mover_best = float(np.mean(ms_best))

# =================== Summary & Save ===================
summary = {
    "count": len(items),
    # ROUGE
    "rouge1_ref1_mean": rouge_ref1["rouge1"],
    "rouge2_ref1_mean": rouge_ref1["rouge2"],
    "rougeL_ref1_mean": rouge_ref1["rougeL"],
    "rougeLsum_ref1_mean": rouge_ref1["rougeLsum"],
    "rouge1_ref2_mean": rouge_ref2["rouge1"] if rouge_ref2["rouge1"] is not None else None,
    "rouge2_ref2_mean": rouge_ref2["rouge2"] if rouge_ref2["rouge2"] is not None else None,
    "rougeL_ref2_mean": rouge_ref2["rougeL"] if rouge_ref2["rougeL"] is not None else None,
    "rougeLsum_ref2_mean": rouge_ref2["rougeLsum"] if rouge_ref2["rougeLsum"] is not None else None,
    "rouge1_best_over_refs": float(np.mean(rouge1_list)),
    "rouge2_best_over_refs": float(np.mean(rouge2_list)),
    "rougeL_best_over_refs": float(np.mean(rougeL_list)),
    "rougeLsum_best_over_refs": float(np.mean(rougeLsum_list)),
    # BLEU / chrF
    "sacrebleu_multi_ref": bleu_multi,
    "sacrebleu_ref1": bleu_r1,
    "sacrebleu_ref2": bleu_r2,
    "sacrebleu_avg": None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0,
    "chrf_multi_ref": chrf_multi,
    "chrf_ref1": chrf_r1,
    "chrf_ref2": chrf_r2,
    "chrf_avg": None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0,
    # METEOR
    "meteor_ref1": met_r1,
    "meteor_ref2": met_r2,
    "meteor_best_over_refs": None if 'met_best' not in locals() else met_best,
    # BERTScore
    "bertscore_ref1": {"P": bP1, "R": bR1, "F1": bF1},
    "bertscore_ref2": ({"P": bP2, "R": bR2, "F1": bF2} if bF2 is not None else None),
    "bertscore_best_over_refs": {"P": bP_best, "R": bR_best, "F1": bF_best},
    # MoverScore
    "moverscore_ref1": mover_ref1,
    "moverscore_ref2": mover_ref2,
    "moverscore_best_over_refs": mover_best,
    # Misc
    "preds_path": PRED_PATH,
    "refs_path": TEST_PATH,
}

rows = []
for i in range(len(items)):
    row = {
        "id": items[i].get("id", i),
        "title": items[i].get("title", ""),
        "prediction": preds[i],
        "ref1": ref1[i],
        "ref2": ref2[i],
        "rougeLsum_best": rougeLsum_list[i],
        "best_ref": "Ref1" if (best_ref_ix[i]==0 or not ref2[i]) else "Ref2",
        "moverscore_best": ms_best[i],
    }
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, "eval_report_per_sample.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "eval_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n=== Overall Metrics ===")
for k,v in summary.items():
    if k in ["preds_path","refs_path","count"]: continue
    print(f"{k}: {v}")

print(f"\n📄 Saved per-sample CSV -> {csv_path}")
print(f"📊 Saved summary JSON   -> {os.path.join(OUT_DIR, 'eval_summary.json')}")
print("✅ Done.")


In [ ]:
# =================== عرض كل العينات (ID / Title / Prediction / Ref1 / Ref2) ===================
import os, glob, pandas as pd
from typing import List, Dict, Any

TOP_N = 3
OUT_ROOT = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/test_eval_UNIFIED"  # adjust if needed

def _score_rec(rec: Dict[str, Any]) -> float:
    sc = (rec.get("gen_info") or {}).get("score")
    try:
        return float(sc)
    except Exception:
        return float(len(str(rec.get("prediction", ""))))  # fallback: length

def _print_one(idx:int, rec:Dict[str,Any], score:float):
    refs = rec.get("references") or []
    print(f"\n{'='*100}")
    print(f"#{idx} | ID: {rec.get('id')} | Strategy: {(rec.get('gen_info') or {}).get('strategy')} | Score: {score:.4f}")
    title = str(rec.get("title",""))
    if title: print(f"Title: {title[:120]}")
    print("\nPrediction:\n", rec.get("prediction",""))
    if len(refs)>=1 and isinstance(refs[0], str) and refs[0].strip():
        print("\nRef1:\n", refs[0])
    if len(refs)>=2 and isinstance(refs[1], str) and refs[1].strip():
        print("\nRef2:\n", refs[1])

def top_n_from_records(records: List[Dict[str, Any]], n:int=3):
    scored = [(_score_rec(r), r) for r in records]
    scored.sort(key=lambda x: x[0], reverse=True)
    for i, (sc, rec) in enumerate(scored[:n], 1):
        _print_one(i, rec, sc)

def top_n_from_csv(out_root:str, n:int=3):
    runs = sorted(glob.glob(os.path.join(out_root, "run_*")), key=os.path.getmtime, reverse=True)
    for rd in runs:
        csv_path = os.path.join(rd, "test_preds_refs_metrics.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            # use rerank_score if available else prediction length
            if "rerank_score" in df.columns:
                df["__score__"] = pd.to_numeric(df["rerank_score"], errors="coerce")
            else:
                df["__score__"] = None
            df["__score__"] = df["__score__"].fillna(df["prediction"].astype(str).str.len().astype(float))
            df = df.sort_values("__score__", ascending=False).head(n)
            for i, row in enumerate(df.itertuples(index=False), 1):
                rec = {
                    "id": getattr(row, "id", None),
                    "title": getattr(row, "title", ""),
                    "prediction": getattr(row, "prediction", ""),
                    "references": [getattr(row, "ref1", "") , getattr(row, "ref2", "")],
                    "gen_info": {"strategy": getattr(row, "strategy", None)}
                }
                _print_one(i, rec, float(getattr(row, "__score__", 0.0)))
            print(f"\n(From {csv_path})")
            return
    print("No CSV found under OUT_ROOT. Set OUT_ROOT correctly or run generation first.")

# --- Use in-memory 'records' if available; else load latest CSV ---
try:
    records  # will raise NameError if not defined
    print("Using in-memory 'records'")
    top_n_from_records(records, TOP_N)
except NameError:
    print("No in-memory 'records' found; scanning OUT_ROOT for latest run…")
    top_n_from_csv(OUT_ROOT, TOP_N)

In [ ]:
!pip install -q xlsxwriter

In [ ]:
# =================== تنزيل IDs + التوليد + المراجع كملفات إلى جهازك ===================
import os, json, re
from typing import List, Dict, Any
import pandas as pd

# --- مساراتك (عدّل إذا لزم) ---
EVAL_CSV  = "/content/drive/MyDrive/eval_runs/run_20250925_021123/eval_report_per_sample.csv"  # إن وُجد يكفي
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
PRED_PATH = "/content/drive/MyDrive/mt5_xlsum444_patch_6144/test_eval_ENHANCED_20250924_231254/enhanced_predictions.csv"

# --- أدوات مساعدة ---
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)

def sanitize_text(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith("```"): continue
            if s.startswith("`"): s = s[1:]
            s = s.rstrip("`,")
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj)
            except json.JSONDecodeError:
                pass
    return recs

def load_predictions(path: str, test_df: pd.DataFrame) -> pd.DataFrame:
    """إرجاع DataFrame فيه id + prediction (مزالة الضوضاء)؛ يحاول المزج بالـid وإلا بالترتيب."""
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
        pred_col = None
        for c in ["prediction","pred","summary","generated","generated_summary","output"]:
            if c in df.columns:
                pred_col = c; break
        if pred_col is None:
            raise ValueError(f"لم أجد عمود التنبؤات في CSV. الأعمدة: {list(df.columns)}")
        df["prediction"] = df[pred_col].astype(str).map(sanitize_text)
        if "id" in df.columns:
            return df[["id","prediction"]]
        else:
            df = df.reset_index().rename(columns={"index":"_row"})
            t2 = test_df.reset_index().rename(columns={"index":"_row"})
            merged = pd.merge(t2, df[["_row","prediction"]], on="_row", how="left").drop(columns=["_row"])
            return merged[["id","prediction"]]
    else:
        # JSON/JSONL
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict) and "records" in data:
                data = data["records"]
        except Exception:
            data = load_jsonl(path)
        rows = []
        for obj in data:
            pred = sanitize_text(obj.get("prediction") or obj.get("summary") or obj.get("pred") or "")
            rows.append({"id": obj.get("id"), "prediction": pred})
        df = pd.DataFrame(rows)
        if "id" in df.columns and df["id"].notna().all():
            return df[["id","prediction"]]
        else:
            df = df.reset_index().rename(columns={"index":"_row"})
            t2 = test_df.reset_index().rename(columns={"index":"_row"})
            merged = pd.merge(t2, df[["_row","prediction"]], on="_row", how="left").drop(columns=["_row"])
            return merged[["id","prediction"]]

# --- 1) جهِّز DataFrame من التست (للمراجع) ---
items = load_jsonl(TEST_PATH)
test_df = pd.DataFrame([{
    "id": it.get("id"),
    "title": it.get("title",""),
    "ref1": sanitize_text(it.get("target_summary","")),
    "ref2": sanitize_text(it.get("target_summary_2",""))
} for it in items])

# --- 2) إمّا نقرأ من تقرير التقييم مباشرة، أو ندمج من التست + التوليد ---
if os.path.exists(EVAL_CSV):
    df_eval = pd.read_csv(EVAL_CSV)
    # تأمين الأعمدة
    for c in ["id","title","prediction","ref1","ref2"]:
        if c not in df_eval.columns:
            df_eval[c] = ""
    merged = df_eval[["id","title","prediction","ref1","ref2"]].copy()
else:
    # دمج من التست + التوليد
    pred_df = load_predictions(PRED_PATH, test_df)
    if "id" in pred_df.columns:
        merged = pd.merge(test_df, pred_df, on="id", how="left")
    else:
        # تم الدمج بالترتيب داخل load_predictions
        merged = pd.concat([test_df[["id","title","ref1","ref2"]], pred_df[["prediction"]]], axis=1)

# --- 3) تنظيف نهائي وترتيب ---
for c in ["title","prediction","ref1","ref2"]:
    merged[c] = merged[c].astype(str).map(sanitize_text)
merged = merged[["id","title","prediction","ref1","ref2"]].sort_values(by="id", kind="stable").reset_index(drop=True)

print(f"✅ Prepared {len(merged)} rows")

# --- 4) حفظ في /content وتنزيل إلى جهازك ---
CSV_OUT  = "/content/ids_preds_refs.csv"
XLSX_OUT = "/content/ids_preds_refs.xlsx"

merged.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(XLSX_OUT, engine="xlsxwriter") as writer:
    merged.to_excel(writer, index=False, sheet_name="data")

print("📄 Saved CSV  ->", CSV_OUT)
print("📊 Saved XLSX ->", XLSX_OUT)

# تنزيل لسطح جهازك (Colab)
try:
    from google.colab import files
    files.download(CSV_OUT)
    files.download(XLSX_OUT)
except Exception as e:
    print("ℹ️ لم يتم استخدام Google Colab، بإمكانك تحميل الملفين يدويًا من المسارات أعلاه.", e)


In [ ]:
# =================== Averages + Top/Bottom-3 for ALL metrics ===================
import os, json
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ---- مسار تقريرك لكل عيّنة ----
EVAL_CSV = "/content/drive/MyDrive/eval_runs/run_20250925_021123/eval_report_per_sample.csv"
assert os.path.exists(EVAL_CSV), f"لم أجد الملف: {EVAL_CSV}"

# ---- قراءة التقرير ----
df = pd.read_csv(EVAL_CSV)

# أعمدة نصية يُستثنى منها (قد تختلف من بيئة لأخرى)
TEXT_COLS = {
    "title","prediction","ref1","ref2","best_ref_used",
    "bertscore_best_ref_text","moverscore_best_ref_text","strategy"
}
ID_COLS = {"id", "index", "_row"}

# اختيار كل الأعمدة الرقمية كمقاييس، مع استبعاد الأعمدة المعروفة أعلاه
metric_cols = []
for c in df.columns:
    if c in TEXT_COLS or c in ID_COLS:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        # استبعاد أعمدة قد تكون طول/عداد داخلي إن وجدت
        metric_cols.append(c)

if not metric_cols:
    raise ValueError("لم أجد أعمدة رقمية للمقاييس في التقرير.")

# ---- ملخّص المتوسطات/الاحصاءات لكل مقياس ----
stats_rows = []
for m in metric_cols:
    s = df[m]
    s_valid = s.dropna()
    stats_rows.append({
        "metric": m,
        "count": int(s_valid.shape[0]),
        "mean": float(s_valid.mean()) if len(s_valid) else np.nan,
        "std": float(s_valid.std(ddof=0)) if len(s_valid) else np.nan,
        "min": float(s_valid.min()) if len(s_valid) else np.nan,
        "max": float(s_valid.max()) if len(s_valid) else np.nan,
    })
stats_df = pd.DataFrame(stats_rows).sort_values("metric").reset_index(drop=True)

print(f"✅ Found {len(metric_cols)} metric columns")
display(stats_df)

# ---- دوال مساعدة للعرض ----
def truncate(s, n=300):
    s = "" if not isinstance(s, str) else s
    return s if len(s) <= n else (s[:n] + "…")

def top_bottom(df, col, k=3):
    d = df[["id","title","prediction","ref1","ref2", col]].copy()
    d = d.dropna(subset=[col])
    # أفضل K (قيم أعلى) وأسوأ K (قيم أدنى)
    topk = d.sort_values(col, ascending=False, kind="stable").head(k)
    botk = d.sort_values(col, ascending=True, kind="stable").head(k)
    # تقصير النصوص الطويلة لعرض أنيق
    for c in ["prediction","ref1","ref2"]:
        if c in topk.columns:
            topk[c] = topk[c].astype(str).map(lambda x: truncate(x, 500))
            botk[c] = botk[c].astype(str).map(lambda x: truncate(x, 500))
    return topk, botk

# ---- طباعة أفضل/أسوأ 3 لكل مقياس في الخرج + بناء تقرير HTML ----
out_dir = os.path.dirname(EVAL_CSV)
avg_csv = os.path.join(out_dir, "metrics_averages.csv")
report_html = os.path.join(out_dir, "metrics_report_top_bottom.html")

# حفظ المتوسطات CSV
stats_df.to_csv(avg_csv, index=False, encoding="utf-8-sig")
print("📊 Saved averages CSV ->", avg_csv)

# بناء HTML مجمّع
html_parts = [
    "<h2>Metrics Averages</h2>",
    stats_df.to_html(index=False, escape=False)
]

for m in metric_cols:
    topk, botk = top_bottom(df, m, k=3)
    print(f"\n=== {m} — Top 3 ===")
    display(topk)

    print(f"=== {m} — Worst 3 ===")
    display(botk)

    html_parts.append(f"<h3>{m} — Top 3</h3>")
    html_parts.append(
        topk.to_html(index=False, escape=False)
    )
    html_parts.append(f"<h3>{m} — Worst 3</h3>")
    html_parts.append(
        botk.to_html(index=False, escape=False)
    )

# حفظ HTML
with open(report_html, "w", encoding="utf-8") as f:
    f.write("""
    <meta charset="utf-8">
    <style>
      table {border-collapse: collapse; width: 100%;}
      th, td {border: 1px solid #ddd; padding: 6px; vertical-align: top;}
      th {background: #f2f2f2;}
    </style>
    """ + "\n".join(html_parts))

print("📄 Saved HTML report ->", report_html)

# (اختياري) تنزيل الملفين في كولاب
try:
    from google.colab import files
    files.download(avg_csv)
    files.download(report_html)
except Exception:
    pass


In [ ]:
# =================== Best/Worst Overall (normalize-all-metrics) + full texts ===================
import os, pandas as pd, numpy as np, json
from IPython.display import display, HTML

# مسار تقريرك لكل عينة
EVAL_CSV = "/content/drive/MyDrive/eval_runs/run_20250925_021123/eval_report_per_sample.csv"
assert os.path.exists(EVAL_CSV), f"لم أجد الملف: {EVAL_CSV}"

# 1) قراءة التقرير
df = pd.read_csv(EVAL_CSV)

# 2) تحديد الأعمدة النصية/المُعرّفات لاستبعادها من التجميع
TEXT_COLS = {
    "title","prediction","ref1","ref2","best_ref_used",
    "bertscore_best_ref_text","moverscore_best_ref_text","strategy"
}
ID_COLS = {"id","index","_row"}
INCLUDE_PATTERNS = ["rouge", "sacrebleu", "chrf", "meteor", "bertscore", "moverscore"]

# كل الأعمدة الرقمية التي تطابق أنماط المقاييس
metric_cols = []
for c in df.columns:
    if c in TEXT_COLS or c in ID_COLS:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        name_l = c.lower()
        if any(p in name_l for p in INCLUDE_PATTERNS):
            metric_cols.append(c)

if not metric_cols:
    raise ValueError("لم أجد أعمدة رقمية للمقاييس في التقرير.")

# 3) تطبيع كل مقياس إلى [0,1] باستخدام min-max (مع معالجة NaN)
norm = pd.DataFrame(index=df.index)
for c in metric_cols:
    s = df[c].astype(float)
    # عالج NaN (اعتبرها أدنى قيمة لعدم مكافأة النتائج الناقصة)
    s = s.fillna(s.min())
    denom = s.max() - s.min()
    if denom <= 1e-12:
        norm[c] = 0.0  # كله متساوٍ
    else:
        norm[c] = (s - s.min()) / denom

# 4) درجة تجميعية (متوسط بسيط لكل المقاييس بعد التطبيع)
df["agg_score"] = norm.mean(axis=1)

# 5) اختيار الأفضل والأسوأ
best_idx  = int(df["agg_score"].idxmax())
worst_idx = int(df["agg_score"].idxmin())

best_row  = df.loc[best_idx].copy()
worst_row = df.loc[worst_idx].copy()

# 6) تجهيز عرض كامل للنصوص + إظهار كل قيم المقاييس لهاتين العينتين
def row_view(r: pd.Series) -> pd.DataFrame:
    cols_show = ["id","title","prediction","ref1","ref2","agg_score"] + metric_cols
    out = pd.DataFrame(r[cols_show]).reset_index()
    out.columns = ["field","value"]
    return out

pd.set_option("display.max_colwidth", None)  # نص كامل بدون قص
pd.set_option("display.max_rows", 10000)

print(f"✅ Metrics used ({len(metric_cols)}):", ", ".join(metric_cols))

print("\n🌟 Best Overall (highest agg_score):")
display(row_view(best_row))

print("\n⚠️ Worst Overall (lowest agg_score):")
display(row_view(worst_row))

# 7) حفظ ملفات
out_dir = os.path.dirname(EVAL_CSV)
best_json  = os.path.join(out_dir, "best_overall.json")
worst_json = os.path.join(out_dir, "worst_overall.json")
best_html  = os.path.join(out_dir, "best_overall.html")
worst_html = os.path.join(out_dir, "worst_overall.html")

with open(best_json, "w", encoding="utf-8") as f:
    json.dump(best_row.to_dict(), f, ensure_ascii=False, indent=2)
with open(worst_json, "w", encoding="utf-8") as f:
    json.dump(worst_row.to_dict(), f, ensure_ascii=False, indent=2)

# HTML بسيط مع إبقاء النصوص كاملة
def to_html_table(r: pd.Series, title=""):
    dfv = row_view(r)
    return f"<h2>{title}</h2>" + dfv.to_html(index=False, escape=False)

with open(best_html, "w", encoding="utf-8") as f:
    f.write('<meta charset="utf-8">')
    f.write(to_html_table(best_row, "Best Overall"))
with open(worst_html, "w", encoding="utf-8") as f:
    f.write('<meta charset="utf-8">')
    f.write(to_html_table(worst_row, "Worst Overall"))

print("\n📄 Saved:")
print(" -", best_json)
print(" -", worst_json)
print(" -", best_html)
print(" -", worst_html)

# (اختياري) تنزيل الملفات في كولاب
try:
    from google.colab import files
    files.download(best_json)
    files.download(worst_json)
    files.download(best_html)
    files.download(worst_html)
except Exception:
    pass


In [ ]:
# =================== Top-5 / Bottom-5 (normalize-all-metrics) + full texts ===================
import os, pandas as pd, numpy as np, json
from IPython.display import display

# مسار تقريرك لكل عينة
EVAL_CSV = "/content/drive/MyDrive/eval_runs/run_20250925_021123/eval_report_per_sample.csv"
assert os.path.exists(EVAL_CSV), f"لم أجد الملف: {EVAL_CSV}"

# 1) قراءة التقرير
df = pd.read_csv(EVAL_CSV)

# 2) أعمدة نصية/معرّفات لاستبعادها من التجميع
TEXT_COLS = {
    "title","prediction","ref1","ref2","best_ref_used",
    "bertscore_best_ref_text","moverscore_best_ref_text","strategy"
}
ID_COLS = {"id","index","_row"}
INCLUDE_PATTERNS = ["rouge", "sacrebleu", "chrf", "meteor", "bertscore", "moverscore"]

# أعمدة المقاييس الرقمية
metric_cols = []
for c in df.columns:
    if c in TEXT_COLS or c in ID_COLS:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        if any(p in c.lower() for p in INCLUDE_PATTERNS):
            metric_cols.append(c)

if not metric_cols:
    raise ValueError("لم أجد أعمدة رقمية للمقاييس في التقرير.")

# 3) تطبيع كل مقياس إلى [0,1] (min-max) مع معالجة NaN كأدنى قيمة
norm = pd.DataFrame(index=df.index)
for c in metric_cols:
    s = df[c].astype(float)
    s = s.fillna(s.min())
    denom = s.max() - s.min()
    norm[c] = 0.0 if denom <= 1e-12 else (s - s.min()) / denom

# 4) درجة تجميعية (متوسط بسيط بعد التطبيع)
df["agg_score"] = norm.mean(axis=1)

# 5) اختيار أفضل/أسوأ 5
TOP_K = 5
topk  = df.sort_values("agg_score", ascending=False, kind="stable").head(TOP_K).copy()
botk  = df.sort_values("agg_score", ascending=True,  kind="stable").head(TOP_K).copy()

# 6) دالة عرض صف بالكامل
def row_view(r: pd.Series) -> pd.DataFrame:
    cols_show = ["id","title","prediction","ref1","ref2","agg_score"] + metric_cols
    cols_show = [c for c in cols_show if c in r.index]
    out = pd.DataFrame(r[cols_show]).reset_index()
    out.columns = ["field","value"]
    return out

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 10000)

print(f"✅ Metrics used ({len(metric_cols)}): {', '.join(metric_cols)}")
print("🌟 Top-5 IDs:", list(topk["id"].values))
print("⚠️ Bottom-5 IDs:", list(botk["id"].values))

print("\n==================== TOP-5 (Best Overall) ====================")
for _, r in topk.iterrows():
    print(f"\n--- ID: {r['id']} | agg_score: {r['agg_score']:.4f} ---")
    display(row_view(r))

print("\n==================== BOTTOM-5 (Worst Overall) ====================")
for _, r in botk.iterrows():
    print(f"\n--- ID: {r['id']} | agg_score: {r['agg_score']:.4f} ---")
    display(row_view(r))

# 7) حفظ ملفات
out_dir = os.path.dirname(EVAL_CSV)
top_jsonl = os.path.join(out_dir, "top5_overall.jsonl")
bot_jsonl = os.path.join(out_dir, "bottom5_overall.jsonl")
top_csv   = os.path.join(out_dir, "top5_overall_rows.csv")
bot_csv   = os.path.join(out_dir, "bottom5_overall_rows.csv")
top_html  = os.path.join(out_dir, "top5_overall.html")
bot_html  = os.path.join(out_dir, "bottom5_overall.html")

# JSONL
with open(top_jsonl, "w", encoding="utf-8") as f:
    for _, r in topk.iterrows():
        json.dump(r.to_dict(), f, ensure_ascii=False); f.write("\n")
with open(bot_jsonl, "w", encoding="utf-8") as f:
    for _, r in botk.iterrows():
        json.dump(r.to_dict(), f, ensure_ascii=False); f.write("\n")

# CSV
topk.to_csv(top_csv, index=False, encoding="utf-8-sig")
botk.to_csv(bot_csv, index=False, encoding="utf-8-sig")

# HTML مبسّط (كل صف كجدول)
def to_html_table(r: pd.Series, title=""):
    dfv = row_view(r)
    return f"<h3>{title} — ID {r['id']} (agg {r['agg_score']:.4f})</h3>" + dfv.to_html(index=False, escape=False)

with open(top_html, "w", encoding="utf-8") as f:
    f.write('<meta charset="utf-8"><h2>TOP-5 (Best Overall)</h2>')
    for _, r in topk.iterrows():
        f.write(to_html_table(r))
with open(bot_html, "w", encoding="utf-8") as f:
    f.write('<meta charset="utf-8"><h2>BOTTOM-5 (Worst Overall)</h2>')
    for _, r in botk.iterrows():
        f.write(to_html_table(r))

print("\n📄 Saved:")
for p in [top_jsonl, bot_jsonl, top_csv, bot_csv, top_html, bot_html]:
    print(" -", p)

# (اختياري) تنزيل في كولاب
try:
    from google.colab import files
    for p in [top_jsonl, bot_jsonl, top_csv, bot_csv, top_html, bot_html]:
        files.download(p)
except Exception:
    pass
